In [138]:
import warnings
warnings.filterwarnings('ignore')

import os
import gc
import pickle

import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt

from datetime import datetime
from pandas.tseries.offsets import MonthEnd

input_table = 'TRN_DF_ECOM_OFFTAKE_CHAIN_PSKU'
month_run = '2026-06-30'
os.listdir('/data/aman_singh/acuuracy_check')

['missing_keys_drm.csv',
 'seasonality_all2.csv',
 'chek_nan.csv',
 'missing_keys_drm2.csv',
 'all_combination_ecom_may_pred.csv',
 'QCOM Chain FC PSKU Primary_as_on_11th_Feb_2026.xlsb',
 'soh_recent_qcom.csv',
 'all_combination_GT_live_june.csv',
 'combine_model+missing_forecasts_brand_asm.ipynb',
 'April-26 Plans.xlsx',
 'QCOM Chain PSKU OTP Output',
 'Heuristics_all_combination_ecom_may_live.xlsx',
 'Heuristics_all_combination_qcom_chain_psku_june_live.xlsx',
 'Norms 202602.csv',
 'Norms 202606.csv',
 'duplicates_after_realignment.csv',
 'ALL Channels Accuracy_fva.ipynb',
 'QCOM Chain Depot PSKU Primary_Live run_03_Jun_2026.csv',
 'all_combination_qcom_cp_july_pred.csv',
 't_thres_df_2.csv',
 'ecom_chain_psku_offtake_to_secondary_v6_PROD.ipynb',
 'qcom_chain_depot_psku_primary_forecast.csv',
 "gt_channels Live Run may'26.csv",
 'seasonality.xlsx',
 'missing_df_gt_all.csv',
 'plan_actuals_aggregated2.csv',
 'ECOM Chain PSKU Primary_as_on_12_Jan_2026 (1).xlsb',
 'Heuristics_all_combin

In [139]:
base_dir = '/data/aman_singh/acuuracy_check'

In [140]:
def list_all_files_in_directory(root):
    out = []

    for path, subdirs, files in os.walk(root):
        for name in files:
            out.append(os.path.join(path, name))

    return out

In [141]:
list_all_files_in_directory(base_dir)

['/data/aman_singh/acuuracy_check/missing_keys_drm.csv',
 '/data/aman_singh/acuuracy_check/seasonality_all2.csv',
 '/data/aman_singh/acuuracy_check/chek_nan.csv',
 '/data/aman_singh/acuuracy_check/missing_keys_drm2.csv',
 '/data/aman_singh/acuuracy_check/all_combination_ecom_may_pred.csv',
 '/data/aman_singh/acuuracy_check/QCOM Chain FC PSKU Primary_as_on_11th_Feb_2026.xlsb',
 '/data/aman_singh/acuuracy_check/soh_recent_qcom.csv',
 '/data/aman_singh/acuuracy_check/all_combination_GT_live_june.csv',
 '/data/aman_singh/acuuracy_check/combine_model+missing_forecasts_brand_asm.ipynb',
 '/data/aman_singh/acuuracy_check/April-26 Plans.xlsx',
 '/data/aman_singh/acuuracy_check/Heuristics_all_combination_ecom_may_live.xlsx',
 '/data/aman_singh/acuuracy_check/Heuristics_all_combination_qcom_chain_psku_june_live.xlsx',
 '/data/aman_singh/acuuracy_check/Norms 202602.csv',
 '/data/aman_singh/acuuracy_check/Norms 202606.csv',
 '/data/aman_singh/acuuracy_check/duplicates_after_realignment.csv',
 '/da

In [142]:
def discover_channel(file_path):
    # file_path = file_path.split('/')

    # if 'ECOM' in file_path:
    #     return 'ECOM'
    # elif 'QCOM' in file_path:
    #     return 'QCOM'
    # elif 'MT' in file_path:
    #     return 'MT'
    # else:
    #     return 'Channel not found'

    return 'ECOM'


In [143]:
from maricovault.MaricoDB import MaricoSnowflake

def get_dbconnection(db_name):    

    KEY_VAULT_NAME = "prod-pwd"

    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'
    

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection


def read_qtr_ind_rate_table():
    """
    Fetch the club sku information from  DWH_SAP_INDEX_TURNOVER_MONTHWISE table.

    Return:
        qtr_ind_rate_data: pandas dataframe
        - dataframe contains all the results from the index rate table.
    """
    connection = get_dbconnection(db_name='PROD')
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=connection, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    connection.close()
    return qtr_ind_rate


dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


In [144]:
data_query = f"""
    select * from {input_table}
    where month_date >= '2023-01-01' and run_month = '{month_run}'
        
"""

offtake_df = pd.read_sql(data_query, dev_conn)
offtake_df.head()

,MONTH_DATE,PLATFORM_NAME,PARENT_MATERIAL_CODE,BRAND_CODE,VOL_IN_RUM,INDEXBPM,IMPUTED,BIG_BILLION_DAYS,BIG_BILLION_DAYS_LAG_1,BIG_BILLION_DAYS_LAG_2,BIG_BILLION_DAYS_LEAD_1,BIG_BILLION_DAYS_LEAD_2,GREAT_INDIAN_FESTIVAL,GREAT_INDIAN_FESTIVAL_LAG_1,GREAT_INDIAN_FESTIVAL_LAG_2,GREAT_INDIAN_FESTIVAL_LEAD_1,GREAT_INDIAN_FESTIVAL_LEAD_2,RUN_MONTH
0,2023-01-31,Amazon ARIPL,718288,SAFF GOLD,9.640,13.27071,0,0,0,0,0,0,0,0,0,0,0,2026-06-30
1,2023-02-28,Amazon ARIPL,718288,SAFF GOLD,7.915,10.89602,0,0,0,0,0,0,0,0,0,0,0,2026-06-30
2,2023-03-31,Amazon ARIPL,718288,SAFF GOLD,9.505,13.08486,0,0,0,0,0,0,0,0,0,0,0,2026-06-30
3,2023-04-30,Amazon ARIPL,718288,SAFF GOLD,9.290,12.78889,0,0,0,0,0,0,0,0,0,0,0,2026-06-30
4,2023-05-31,Amazon ARIPL,718288,SAFF GOLD,8.050,11.08187,0,0,0,0,0,0,0,0,0,0,0,2026-06-30


In [145]:
offtake_df.columns = offtake_df.columns.str.lower()

In [146]:
offtake_df.duplicated(
    subset=['platform_name','parent_material_code', 'month_date']).sum()

0

In [147]:
offtake_df['brand_code'] = np.where(
    ((offtake_df['parent_material_code'] == 715096) &
    (offtake_df['brand_code'] == 'CO_SO_PCP')),
    'CO_SO_FS',
    offtake_df['brand_code']
)

In [148]:
# offtake_df = offtake_df[offtake_df['platform_name'].isin(
#     ['Amazon', 'Big Basket', 'Flipkart Grocery', 'Flipkart National'])]

In [149]:
offtake_df['key'] = offtake_df[['platform_name','parent_material_code']].astype(str).agg('_'.join, axis=1)
# offtake_df.rename(columns={'realigned_psku': 'parent_material_code'}, inplace=True)
# offtake_df.drop([ 'run_month'], axis=1, inplace=True)
offtake_df['parent_material_code'] = offtake_df['parent_material_code'].astype(int)

In [150]:
offtake_df.duplicated(subset=['key', 'month_date']).sum()

0

In [151]:
(offtake_df['key'] == offtake_df[['platform_name','parent_material_code']].astype(str).agg('_'.join, axis=1)).all()

True

In [152]:
# realigned_df.to_csv('OT_data_debug.csv', index=False)

### Collate MIL

In [153]:
base_dir

'/data/aman_singh/acuuracy_check'

In [154]:
def collate_file(file_hint, extension='.csv'):
    collated_file = pd.DataFrame()

    run_path = f'{base_dir}'
    all_files = list_all_files_in_directory(run_path)

    for file_path in all_files:
        if file_hint in file_path:
            if extension == '.csv':
                print(file_path)
                read_file = pd.read_csv(file_path)
                # read_file['channel'] = discover_channel(file_path)
                read_file['run'] = 'run'
                read_file['step'] = file_path.split('/')[3]
                read_file['file_path'] = file_path

                collated_file = pd.concat(
                    [collated_file, read_file]
                )
                del read_file

    return collated_file

In [155]:
trend_file_df = collate_file('trend_file_train_till')
prophet_file_df = collate_file('prophet_data_train_till')

/data/aman_singh/acuuracy_check/trend_file_train_till_31_May_2026 (9).csv
/data/aman_singh/acuuracy_check/trend_file_train_till_31_May_2026 (8).csv
/data/aman_singh/acuuracy_check/prophet_data_train_till_31_May_2026 (9).csv
/data/aman_singh/acuuracy_check/prophet_data_train_till_31_May_2026 (8).csv


In [156]:
# forecast_train_till_file_df = collate_file('forecast_train_till_')

In [157]:
# forecast_train_till_file_df

In [158]:
trend_file_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,vol_in_rum,brand_code,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2
0,Amazon ARIPL_718288,2023-01-31,3.977277e-01,9.020000,9.154167,9.113322,9.397314,5.523056e-03,0.125256,0.12712,0.126552,0.130496,718288,Amazon ARIPL,9.640,SAFF GOLD,0.0,0.0,0.0,0.0,0.0,138865.260689,0.133866,NaN,NaN,9.640,0.133866,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN
1,Amazon ARIPL_718288,2023-02-28,1.003744e+01,9.020000,9.154167,7.401517,8.737608,1.393852e-01,0.125256,0.12712,0.102781,0.121335,718288,Amazon ARIPL,7.915,SAFF GOLD,0.0,0.0,0.0,0.0,0.0,138865.260689,0.109912,NaN,NaN,7.915,0.109912,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN
2,Amazon ARIPL_718288,2023-03-31,9.856017e+00,9.020000,9.154167,12.787684,9.508410,1.368658e-01,0.125256,0.12712,0.177577,0.132039,718288,Amazon ARIPL,9.505,SAFF GOLD,0.0,0.0,0.0,0.0,0.0,138865.260689,0.131991,NaN,NaN,9.505,0.131991,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN
3,Amazon ARIPL_718288,2023-04-30,9.579111e+00,9.020000,9.154167,4.585437,9.076793,1.330206e-01,0.125256,0.12712,0.063676,0.126045,718288,Amazon ARIPL,9.290,SAFF GOLD,0.0,0.0,0.0,0.0,0.0,138865.260689,0.129006,NaN,NaN,9.290,0.129006,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN
4,Amazon ARIPL_718288,2023-05-31,9.810191e+00,8.903333,9.154167,8.930346,8.285612,1.362295e-01,0.123636,0.12712,0.124011,0.115058,718288,Amazon ARIPL,8.050,SAFF GOLD,0.0,0.0,0.0,0.0,0.0,138865.260689,0.111787,NaN,NaN,8.050,0.111787,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85551,Nykaa_810605,2026-09-30,1.483873e-81,0.000000,0.000000,0.000000,0.000000,1.819784e-85,0.000000,0.00000,0.000000,0.000000,810605,Nykaa,0.000,KAYA_ML,NaN,NaN,NaN,NaN,NaN,1226.374229,0.000000,NaN,NaN,0.000,0.000000,2026-05-31,1.980353,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0
85552,Nykaa_810605,2026-10-31,1.421690e-81,0.000000,0.000000,0.000000,0.132000,1.743524e-85,0.000000,0.00000,0.000000,0.000016,810605,Nykaa,0.000,KAYA_ML,NaN,NaN,NaN,NaN,NaN,1226.374229,0.000000,NaN,NaN,0.000,0.000000,2026-05-31,1.980353,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0
85553,Nykaa_810605,2026-11-30,-4.772679e-82,0.000000,0.000000,0.000000,0.000000,-5.853091e-86,0.000000,0.00000,0.000000,0.000000,810605,Nykaa,0.000,KAYA_ML,NaN,NaN,NaN,NaN,NaN,1226.374229,0.000000,NaN,NaN,0.000,0.000000,2026-05-31,1.980353,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0
85554,Nykaa_810605,2026-12-31,-5.848641e-82,0.000000,0.000000,0.000000,0.000000,-7.172623e-86,0.000000,0.00000,0.000000,0.000000,810605,Nykaa,0.000,KAYA_ML,NaN,NaN,NaN,NaN,NaN,1226.374229,0.000000,NaN,NaN,0.000,0.000000,2026-05-31,1.980353,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0


In [159]:
trend_file_df.columns

Index(['key', 'month_date', 'pred_SARIMA', 'pred_p3m', 'pred_p6m',
       'pred_prophet', 'pred_rf', 'pred_value_SARIMA', 'pred_value_p3m',
       'pred_value_p6m', 'pred_value_prophet', 'pred_value_rf',
       'parent_material_code', 'platform_name', 'vol_in_rum', 'brand_code',
       'great_indian_festival', 'great_indian_festival_lag_1',
       'great_indian_festival_lag_2', 'great_indian_festival_lead_1',
       'great_indian_festival_lead_2', 'qtr_ind_rate', 'vol_in_rum_value',
       'pred_best_model', 'pred_value_best_model', 'vol_in_rum_treated',
       'vol_in_rum_value_treated', 'train_till', 'cov', 'run', 'step',
       'file_path', 'big_billion_days', 'big_billion_days_lag_1',
       'big_billion_days_lag_2', 'big_billion_days_lead_1',
       'big_billion_days_lead_2'],
      dtype='object')

In [160]:
trend_file_df['platform_name'].unique()

array(['Amazon ARIPL', 'Amazon RK', 'Big Basket', 'Flipkart Grocery',
       'Flipkart National', 'Meesho', 'Myntra', 'Nykaa'], dtype=object)

In [161]:
trend_file_df['month_date'] = pd.to_datetime(trend_file_df['month_date'])
prophet_file_df['month_date'] = pd.to_datetime(prophet_file_df['month_date'])

trend_file_df['train_till'] = pd.to_datetime(trend_file_df['train_till'])
prophet_file_df['train_till'] = pd.to_datetime(prophet_file_df['train_till'])

trend_file_df['run_month'] = pd.to_datetime(trend_file_df['train_till'] + MonthEnd(1))
prophet_file_df['run_month'] = pd.to_datetime(prophet_file_df['train_till'] + MonthEnd(1))

In [162]:
trend_file_df.duplicated(subset=['key', 'month_date', 'run_month']).sum(), \
prophet_file_df.duplicated(subset=['key', 'month_date', 'run_month']).sum()

(0, 0)

In [163]:
mappings = {}

for run_month in trend_file_df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   

{Timestamp('2026-06-30 00:00:00'): {Timestamp('2026-06-30 00:00:00'): 'M',
  Timestamp('2026-07-31 00:00:00'): 'M+1',
  Timestamp('2026-08-31 00:00:00'): 'M+2',
  Timestamp('2026-09-30 00:00:00'): 'M+3',
  Timestamp('2026-10-31 00:00:00'): 'M+4',
  Timestamp('2026-11-30 00:00:00'): 'M+5',
  Timestamp('2026-12-31 00:00:00'): 'M+6',
  Timestamp('2027-01-31 00:00:00'): 'M+7',
  Timestamp('2027-02-28 00:00:00'): 'M+8'}}

In [164]:
trend_file_df['M month'] = trend_file_df.apply(
    lambda x: mappings[x['run_month']].get(
        x['month_date']
    ), axis=1
)

In [165]:
trend_file_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,vol_in_rum,brand_code,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month
0,Amazon ARIPL_718288,2023-01-31,3.977277e-01,9.020000,9.154167,9.113322,9.397314,5.523056e-03,0.125256,0.12712,0.126552,0.130496,718288,Amazon ARIPL,9.640,SAFF GOLD,0.0,0.0,0.0,0.0,0.0,138865.260689,0.133866,NaN,NaN,9.640,0.133866,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None
1,Amazon ARIPL_718288,2023-02-28,1.003744e+01,9.020000,9.154167,7.401517,8.737608,1.393852e-01,0.125256,0.12712,0.102781,0.121335,718288,Amazon ARIPL,7.915,SAFF GOLD,0.0,0.0,0.0,0.0,0.0,138865.260689,0.109912,NaN,NaN,7.915,0.109912,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None
2,Amazon ARIPL_718288,2023-03-31,9.856017e+00,9.020000,9.154167,12.787684,9.508410,1.368658e-01,0.125256,0.12712,0.177577,0.132039,718288,Amazon ARIPL,9.505,SAFF GOLD,0.0,0.0,0.0,0.0,0.0,138865.260689,0.131991,NaN,NaN,9.505,0.131991,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None
3,Amazon ARIPL_718288,2023-04-30,9.579111e+00,9.020000,9.154167,4.585437,9.076793,1.330206e-01,0.125256,0.12712,0.063676,0.126045,718288,Amazon ARIPL,9.290,SAFF GOLD,0.0,0.0,0.0,0.0,0.0,138865.260689,0.129006,NaN,NaN,9.290,0.129006,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None
4,Amazon ARIPL_718288,2023-05-31,9.810191e+00,8.903333,9.154167,8.930346,8.285612,1.362295e-01,0.123636,0.12712,0.124011,0.115058,718288,Amazon ARIPL,8.050,SAFF GOLD,0.0,0.0,0.0,0.0,0.0,138865.260689,0.111787,NaN,NaN,8.050,0.111787,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85551,Nykaa_810605,2026-09-30,1.483873e-81,0.000000,0.000000,0.000000,0.000000,1.819784e-85,0.000000,0.00000,0.000000,0.000000,810605,Nykaa,0.000,KAYA_ML,NaN,NaN,NaN,NaN,NaN,1226.374229,0.000000,NaN,NaN,0.000,0.000000,2026-05-31,1.980353,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,M+3
85552,Nykaa_810605,2026-10-31,1.421690e-81,0.000000,0.000000,0.000000,0.132000,1.743524e-85,0.000000,0.00000,0.000000,0.000016,810605,Nykaa,0.000,KAYA_ML,NaN,NaN,NaN,NaN,NaN,1226.374229,0.000000,NaN,NaN,0.000,0.000000,2026-05-31,1.980353,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,M+4
85553,Nykaa_810605,2026-11-30,-4.772679e-82,0.000000,0.000000,0.000000,0.000000,-5.853091e-86,0.000000,0.00000,0.000000,0.000000,810605,Nykaa,0.000,KAYA_ML,NaN,NaN,NaN,NaN,NaN,1226.374229,0.000000,NaN,NaN,0.000,0.000000,2026-05-31,1.980353,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,M+5
85554,Nykaa_810605,2026-12-31,-5.848641e-82,0.000000,0.000000,0.000000,0.000000,-7.172623e-86,0.000000,0.00000,0.000000,0.000000,810605,Nykaa,0.000,KAYA_ML,NaN,NaN,NaN,NaN,NaN,1226.374229,0.000000,NaN,NaN,0.000,0.000000,2026-05-31,1.980353,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,M+6


In [166]:
trend_file_df[trend_file_df['M month'].notna()]

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,vol_in_rum,brand_code,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month
41,Amazon ARIPL_718288,2026-06-30,2.412960e+01,27.226,24.622,28.546677,25.025422,3.350764e-01,0.378075,0.341914,0.396414,0.347516,718288,Amazon ARIPL,0.0,SAFF GOLD,0.0,0.0,0.0,0.0,0.0,138865.260689,0.0,25.025422,0.347516,0.0,0.0,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,M
42,Amazon ARIPL_718288,2026-07-31,2.361028e+01,27.226,24.622,27.384622,25.037278,3.278648e-01,0.378075,0.341914,0.380277,0.347681,718288,Amazon ARIPL,0.0,SAFF GOLD,0.0,0.0,0.0,0.0,0.0,138865.260689,0.0,25.037278,0.347681,0.0,0.0,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,M+1
43,Amazon ARIPL_718288,2026-08-31,2.776276e+01,27.226,24.622,27.606416,30.397286,3.855283e-01,0.378075,0.341914,0.383357,0.422113,718288,Amazon ARIPL,0.0,SAFF GOLD,0.0,0.0,0.0,0.0,0.0,138865.260689,0.0,30.397286,0.422113,0.0,0.0,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,M+2
44,Amazon ARIPL_718288,2026-09-30,2.464636e+01,27.226,24.622,29.646981,25.111586,3.422523e-01,0.378075,0.341914,0.411694,0.348713,718288,Amazon ARIPL,0.0,SAFF GOLD,0.0,0.0,0.0,0.0,0.0,138865.260689,0.0,25.111586,0.348713,0.0,0.0,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,M+3
45,Amazon ARIPL_718288,2026-10-31,2.708011e+01,27.226,24.622,32.709076,30.133809,3.760486e-01,0.378075,0.341914,0.454215,0.418454,718288,Amazon ARIPL,0.0,SAFF GOLD,0.0,0.0,0.0,0.0,0.0,138865.260689,0.0,30.133809,0.418454,0.0,0.0,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,M+4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85551,Nykaa_810605,2026-09-30,1.483873e-81,0.000,0.000,0.000000,0.000000,1.819784e-85,0.000000,0.000000,0.000000,0.000000,810605,Nykaa,0.0,KAYA_ML,NaN,NaN,NaN,NaN,NaN,1226.374229,0.0,NaN,NaN,0.0,0.0,2026-05-31,1.980353,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,M+3
85552,Nykaa_810605,2026-10-31,1.421690e-81,0.000,0.000,0.000000,0.132000,1.743524e-85,0.000000,0.000000,0.000000,0.000016,810605,Nykaa,0.0,KAYA_ML,NaN,NaN,NaN,NaN,NaN,1226.374229,0.0,NaN,NaN,0.0,0.0,2026-05-31,1.980353,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,M+4
85553,Nykaa_810605,2026-11-30,-4.772679e-82,0.000,0.000,0.000000,0.000000,-5.853091e-86,0.000000,0.000000,0.000000,0.000000,810605,Nykaa,0.0,KAYA_ML,NaN,NaN,NaN,NaN,NaN,1226.374229,0.0,NaN,NaN,0.0,0.0,2026-05-31,1.980353,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,M+5
85554,Nykaa_810605,2026-12-31,-5.848641e-82,0.000,0.000,0.000000,0.000000,-7.172623e-86,0.000000,0.000000,0.000000,0.000000,810605,Nykaa,0.0,KAYA_ML,NaN,NaN,NaN,NaN,NaN,1226.374229,0.0,NaN,NaN,0.0,0.0,2026-05-31,1.980353,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,M+6


In [167]:
trend_file_df[['run_month', 'train_till']].drop_duplicates().sort_values(by=['run_month'])

,run_month,train_till
0,2026-06-30,2026-05-31


In [168]:
prophet_file_df[['run_month', 'train_till']].drop_duplicates().sort_values(by=['run_month'])

,run_month,train_till
0,2026-06-30,2026-05-31


In [169]:
trend_file_df['M month'].unique()

array([None, 'M', 'M+1', 'M+2', 'M+3', 'M+4', 'M+5', 'M+6', 'M+7'],
      dtype=object)

In [170]:
brand_md_df = pd.read_excel(r"/data/aman_singh/mt_forecast/Brand_metadata.xlsx")

In [171]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    brand_md_df,
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(trend_file_df)

In [172]:
trend_file_df['portfolio'].isna().sum()

0

In [173]:
prophet_file_df[
    ['month_date', 'key', 'run_month']
].duplicated().sum()

0

In [174]:
prophet_file_df

,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,yhat_60_%ile,yhat_70_%ile,yhat_75_%ile,trend_60_%ile,trend_70_%ile,trend_75_%ile,additive_terms,additive_terms_lower,additive_terms_upper,extra_regressors_additive,extra_regressors_additive_lower,extra_regressors_additive_upper,great_indian_festival,great_indian_festival_lower,great_indian_festival_upper,great_indian_festival_lag_1,great_indian_festival_lag_1_lower,great_indian_festival_lag_1_upper,great_indian_festival_lag_2,great_indian_festival_lag_2_lower,great_indian_festival_lag_2_upper,great_indian_festival_lead_1,great_indian_festival_lead_1_lower,great_indian_festival_lead_1_upper,great_indian_festival_lead_2,great_indian_festival_lead_2_lower,great_indian_festival_lead_2_upper,yearly,yearly_lower,yearly_upper,multiplicative_terms,multiplicative_terms_lower,multiplicative_terms_upper,yhat,key,y,month_date,brand_code,qtr_ind_rate,vol_in_rum,vol_in_rum_value,yhat_value,Model_Run,Model_Type,type,train_till,run,step,file_path,big_billion_days,big_billion_days_lower,big_billion_days_upper,big_billion_days_lag_1,big_billion_days_lag_1_lower,big_billion_days_lag_1_upper,big_billion_days_lag_2,big_billion_days_lag_2_lower,big_billion_days_lag_2_upper,big_billion_days_lead_1,big_billion_days_lead_1_lower,big_billion_days_lead_1_upper,big_billion_days_lead_2,big_billion_days_lead_2_lower,big_billion_days_lead_2_upper,run_month
0,2023-01-31,7.918339,5.331417,13.017822,7.918339,7.918339,9.992578,10.770226,11.250397,7.918339,7.918339,7.918339,1.194983,1.194983,1.194983,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.194983,1.194983,1.194983,0.0,0.0,0.0,9.113322,Amazon ARIPL_718288,9.640,2023-01-31,SAFF GOLD,138865.260689,9.640,0.133866,0.126552,Yes,prophet,training,2026-05-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-30
1,2023-02-28,7.999693,3.430524,10.881236,7.999693,7.999693,7.926816,8.748662,9.127582,7.999693,7.999693,7.999693,-0.598176,-0.598176,-0.598176,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.598176,-0.598176,-0.598176,0.0,0.0,0.0,7.401517,Amazon ARIPL_718288,7.915,2023-02-28,SAFF GOLD,138865.260689,7.915,0.109912,0.102781,Yes,prophet,training,2026-05-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-30
2,2023-03-31,8.089763,9.024279,16.666681,8.089763,8.089763,13.634569,14.499796,14.894958,8.089763,8.089763,8.089763,4.697922,4.697922,4.697922,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.697922,4.697922,4.697922,0.0,0.0,0.0,12.787684,Amazon ARIPL_718288,9.505,2023-03-31,SAFF GOLD,138865.260689,9.505,0.131991,0.177577,Yes,prophet,training,2026-05-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-30
3,2023-04-30,8.176927,0.821601,8.234293,8.176927,8.176927,5.292286,6.055104,6.459645,8.176927,8.176927,8.176927,-3.591490,-3.591490,-3.591490,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.591490,-3.591490,-3.591490,0.0,0.0,0.0,4.585437,Amazon ARIPL_718288,9.290,2023-04-30,SAFF GOLD,138865.260689,9.290,0.129006,0.063676,Yes,prophet,training,2026-05-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-30
4,2023-05-31,8.266997,4.977305,12.750944,8.266997,8.266997,9.813856,10.613436,11.018435,8.266997,8.266997,8.266997,0.663349,0.663349,0.663349,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.663349,0.663349,0.663349,0.0,0.0,0.0,8.930346,Amazon ARIPL_718288,8.050,2023-05-31,SAFF GOLD,138865.260689,8.050,0.111787,0.124011,Yes,prophet,training,2026-05-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,202

In [175]:
# Merge 70th percentile Prophet predictions
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    prophet_file_df[['month_date', 'key', 'run_month', 'yhat_70_%ile','yhat_60_%ile']].rename(
        columns={
            'yhat_70_%ile': 'pred_prophet_70%ile',
            'yhat_60_%ile': 'pred_prophet_60%ile'
        }
    ),
    on=['month_date', 'key', 'run_month'],
    how='left'
)
assert len(trend_file_df) == len_before_merge

In [176]:
assert trend_file_df.duplicated(
    subset=['run_month', 'month_date', 'key']
).sum() == 0

In [177]:
trend_file_df.drop('vol_in_rum', axis=1, inplace=True)

In [178]:
offtake_df.head()

,month_date,platform_name,parent_material_code,brand_code,vol_in_rum,indexbpm,imputed,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,run_month,key
0,2023-01-31,Amazon ARIPL,718288,SAFF GOLD,9.640,13.27071,0,0,0,0,0,0,0,0,0,0,0,2026-06-30,Amazon ARIPL_718288
1,2023-02-28,Amazon ARIPL,718288,SAFF GOLD,7.915,10.89602,0,0,0,0,0,0,0,0,0,0,0,2026-06-30,Amazon ARIPL_718288
2,2023-03-31,Amazon ARIPL,718288,SAFF GOLD,9.505,13.08486,0,0,0,0,0,0,0,0,0,0,0,2026-06-30,Amazon ARIPL_718288
3,2023-04-30,Amazon ARIPL,718288,SAFF GOLD,9.290,12.78889,0,0,0,0,0,0,0,0,0,0,0,2026-06-30,Amazon ARIPL_718288
4,2023-05-31,Amazon ARIPL,718288,SAFF GOLD,8.050,11.08187,0,0,0,0,0,0,0,0,0,0,0,2026-06-30,Amazon ARIPL_718288


In [179]:
assert offtake_df.duplicated(subset=['key', 'month_date']).sum() == 0

In [180]:
offtake_df['month_date'] = pd.to_datetime(offtake_df['month_date'])
offtake_df['run_month'] = pd.to_datetime(offtake_df['run_month'])


In [181]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    offtake_df[['key', 'run_month','month_date', 'vol_in_rum']],
    on=['run_month','month_date', 'key'],
    how='left'
)
assert len(trend_file_df) == len_before_merge

In [182]:
trend_file_df.select_dtypes('number').isna().sum()

pred_SARIMA                        200
pred_p3m                             0
pred_p6m                             0
pred_prophet                         0
pred_rf                              0
pred_value_SARIMA                  200
pred_value_p3m                       0
pred_value_p6m                       0
pred_value_prophet                   0
pred_value_rf                        0
parent_material_code                 0
great_indian_festival            85556
great_indian_festival_lag_1      85556
great_indian_festival_lag_2      85556
great_indian_festival_lead_1     85556
great_indian_festival_lead_2     85556
qtr_ind_rate                         0
vol_in_rum_value                     0
pred_best_model                 106807
pred_value_best_model           106807
vol_in_rum_treated                   0
vol_in_rum_value_treated             0
cov                                  0
big_billion_days                 26083
big_billion_days_lag_1           26083
big_billion_days_lag_2   

In [183]:
trend_file_df.select_dtypes('number').min().round()

pred_SARIMA                     -13577.0
pred_p3m                             0.0
pred_p6m                             0.0
pred_prophet                         0.0
pred_rf                              0.0
pred_value_SARIMA                   -1.0
pred_value_p3m                       0.0
pred_value_p6m                       0.0
pred_value_prophet                   0.0
pred_value_rf                        0.0
parent_material_code            709538.0
great_indian_festival                0.0
great_indian_festival_lag_1          0.0
great_indian_festival_lag_2          0.0
great_indian_festival_lead_1         0.0
great_indian_festival_lead_2         0.0
qtr_ind_rate                         0.0
vol_in_rum_value                     0.0
pred_best_model                   -350.0
pred_value_best_model               -0.0
vol_in_rum_treated                   0.0
vol_in_rum_value_treated             0.0
cov                                  0.0
big_billion_days                     0.0
big_billion_days

In [184]:
trend_file_df['vol_in_rum'].fillna(0, inplace=True)

In [185]:
for col in [ 'pred_best_model', 'pred_value_best_model', 'pred_prophet_70%ile','pred_prophet_60%ile', 'vol_in_rum']:
    trend_file_df[col] = trend_file_df[col].clip(lower=0)

In [186]:
trend_file_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum
0,Amazon ARIPL_718288,2023-01-31,3.977277e-01,9.020000,9.154167,9.113322,9.397314,5.523056e-03,0.125256,0.12712,0.126552,0.130496,718288,Amazon ARIPL,SAFF GOLD,0.0,0.0,0.0,0.0,0.0,138865.260689,0.133866,NaN,NaN,9.640,0.133866,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Saffola Oils,10.770226,9.992578,9.640
1,Amazon ARIPL_718288,2023-02-28,1.003744e+01,9.020000,9.154167,7.401517,8.737608,1.393852e-01,0.125256,0.12712,0.102781,0.121335,718288,Amazon ARIPL,SAFF GOLD,0.0,0.0,0.0,0.0,0.0,138865.260689,0.109912,NaN,NaN,7.915,0.109912,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Saffola Oils,8.748662,7.926816,7.915
2,Amazon ARIPL_718288,2023-03-31,9.856017e+00,9.020000,9.154167,12.787684,9.508410,1.368658e-01,0.125256,0.12712,0.177577,0.132039,718288,Amazon ARIPL,SAFF GOLD,0.0,0.0,0.0,0.0,0.0,138865.260689,0.131991,NaN,NaN,9.505,0.131991,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Saffola Oils,14.499796,13.634569,9.505
3,Amazon ARIPL_718288,2023-04-30,9.579111e+00,9.020000,9.154167,4.585437,9.076793,1.330206e-01,0.125256,0.12712,0.063676,0.126045,718288,Amazon ARIPL,SAFF GOLD,0.0,0.0,0.0,0.0,0.0,138865.260689,0.129006,NaN,NaN,9.290,0.129006,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Saffola Oils,6.055104,5.292286,9.290
4,Amazon ARIPL_718288,2023-05-31,9.810191e+00,8.903333,9.154167,8.930346,8.285612,1.362295e-01,0.123636,0.12712,0.124011,0.115058,718288,Amazon ARIPL,SAFF GOLD,0.0,0.0,0.0,0.0,0.0,138865.260689,0.111787,NaN,NaN,8.050,0.111787,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Saffola Oils,10.613436,9.813856,8.050
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111634,Nykaa_810605,2026-09-30,1.483873e-81,0.000000,0.000000,0.000000,0.000000,1.819784e-85,0.000000,0.00000,0.000000,0.000000,810605,Nykaa,KAYA_ML,NaN,NaN,NaN,NaN,NaN,1226.374229,0.000000,NaN,NaN,0.000,0.000000,2026-05-31,1.980353,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,M+3,Skin Care,0.000000,0.000000,0.000
111635,Nykaa_810605,2026-10-31,1.421690e-81,0.000000,0.000000,0.000000,0.132000,1.743524e-85,0.000000,0.00000,0.000000,0.000016,810605,Nykaa,KAYA_ML,NaN,NaN,NaN,NaN,NaN,1226.374229,0.000000,NaN,NaN,0.000,0.000000,2026-05-31,1.980353,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,M+4,Skin Care,0.000000,0.000000,0.000
111636,Nykaa_810605,2026-11-30,-4.772679e-82,0.000000,0.000000,0.000000,0.000000,-5.853091e-86,0.000000,0.00000,0.000000,0.000000,810605,Nykaa,KAYA_ML,NaN,NaN,NaN,NaN,NaN,1226.374229,0.000000,NaN,NaN,0.000,0.000000,2026-05-31,1.980353,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,M+5,Skin Care,0.000000,0.000000,0.000
111637,Nykaa_810605,2026-12-31,-5.848641e-82,0.000000,0.000000,0.000000,0.000000,-7.172623e-86,0.00000

In [187]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)

In [188]:
trend_file_df['P3M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=3, min_periods=3).mean()

trend_file_df['P6M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=6, min_periods=6).mean()

trend_file_df['LY P3M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=3, min_periods=3).mean()

trend_file_df['LY P6M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=6, min_periods=6).mean()

In [189]:
trend_file_df['LY P3M_copy'] = trend_file_df['LY P3M'].copy()

In [190]:
trend_file_df['P3M Max'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).max()

In [191]:
trend_file_df['P3M Top 2 Mean'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sort(x)[-2:].mean(), raw=True) 

# lambda x: x.nlargest(2).mean(), raw=False

In [192]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)

In [193]:
trend_file_df['MoM P3M growth'] = (
    trend_file_df.groupby(['run_month', 'key'])['P3M']
      .pct_change() * 100
)

In [194]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
trend_file_df['MoM P3M growth_lag_1'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(1)
trend_file_df['MoM P3M growth_lag_2'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(2)



In [195]:
trend_file_df['>=20%_3M_inc_month_count'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['MoM P3M growth'].shift(0)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sum(x >= 20), raw=True) 

In [196]:
trend_file_df['Avg(P3M Mean, Max)'] = trend_file_df[['P3M', 'P3M Max']].mean(axis=1)

In [197]:
for col in ['P3M', 'P6M', 'LY P3M', 'P3M Max', 'P3M Top 2 Mean', 
            'MoM P3M growth', '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)',
            'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2' ]:
    # if not 'LY' in col:  'LY P6M',
    trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
    trend_file_df[col] = trend_file_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )


# for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
#     if not 'LY' in col:
#         trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
#         trend_file_df[col] = trend_file_df.groupby(['key'], as_index = True, group_keys = False)[col].apply(
#             lambda x: x.ffill()
#         )

In [198]:
for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
    trend_file_df[f'{col}_value'] = trend_file_df[col] * trend_file_df['qtr_ind_rate'] / (10 ** 7)

In [199]:
trend_file_df['vol_in_rum_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['vol_in_rum'] / (10 ** 7)
trend_file_df['pred_prophet_70%ile_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['pred_prophet_70%ile'] / (10 ** 7)
trend_file_df['pred_prophet_60%ile_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['pred_prophet_60%ile'] / (10 ** 7)

In [200]:
value_cols = [col for col in trend_file_df.columns if 'value' in col]
value_cols

['pred_value_SARIMA',
 'pred_value_p3m',
 'pred_value_p6m',
 'pred_value_prophet',
 'pred_value_rf',
 'vol_in_rum_value',
 'pred_value_best_model',
 'vol_in_rum_value_treated',
 'P3M_value',
 'P6M_value',
 'LY P3M_value',
 'LY P6M_value',
 'pred_prophet_70%ile_value',
 'pred_prophet_60%ile_value']

In [201]:
for col in value_cols:
    try:
        assert trend_file_df[col].min() >= 0
    except:
        print(col)
    

    # trend_file_df[col] = trend_file_df[col] / (10 ** 7)

pred_value_SARIMA


In [202]:
trend_file_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value
10116,Amazon RK_718472,2023-11-30,0.000000,0.51,1.125,0.452544,0.765,0.000000,0.000025,0.000056,0.000022,0.000038,718472,Amazon RK,ADV-AHO-R,0.0,1.0,0.0,0.0,0.0,496.828458,0.000022,NaN,NaN,0.45,0.000022,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Hair Oils,0.676890,0.547432,0.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000034,0.000027
10117,Amazon RK_718472,2023-12-31,0.450000,0.51,1.125,0.289509,0.774,0.000022,0.000025,0.000056,0.000014,0.000038,718472,Amazon RK,ADV-AHO-R,0.0,0.0,1.0,0.0,0.0,496.828458,0.000000,NaN,NaN,0.00,0.000000,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Hair Oils,0.545380,0.432579,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000027,0.000021
10118,Amazon RK_718472,2024-01-31,0.172500,0.51,1.125,1.143780,1.116,0.000009,0.000025,0.000056,0.000057,0.000055,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000054,NaN,NaN,1.08,0.000054,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Hair Oils,1.380028,1.239281,1.08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000069,0.000062
10119,Amazon RK_718472,2024-02-29,0.678074,0.51,1.125,1.165967,1.296,0.000034,0.000025,0.000056,0.000058,0.000064,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000080,NaN,NaN,1.62,0.000080,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Hair Oils,1.416077,1.293417,1.62,0.51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.51,0.000025,NaN,NaN,NaN,0.000070,0.000064
10120,Amazon RK_718472,2024-03-31,0.844506,0.90,1.125,1.783082,1.818,0.000042,0.000045,0.000056,0.000089,0.000090,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000134,NaN,NaN,2.70,0.000134,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Hair Oils,2.008802,1.887959,2.70,0.90,NaN,NaN,NaN,NaN,NaN,NaN,76.470588,NaN,NaN,NaN,0.90,0.000045,NaN,NaN,NaN,0.000100,0.000094
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33211,Big Basket_719192,2026-09-30,0.000000,0.00,0.000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,719192,Big Basket,VEG_CLEAN,NaN,NaN,NaN,NaN,NaN,100.000000,0.000000,NaN,NaN,0.00,0.000000,2026-05-31,4.605489,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,M+3,Health & Hygiene,0.000000,0.000000,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,-100.000000,-100.0,-100.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000000,0.000000
33212,Big Basket_719192,2026-10-31,0.000000,0.00,0.000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,719192,Big

In [203]:
assert trend_file_df.duplicated(
    subset=['run_month', 'brand_code', 'key', 'month_date']
).sum() == 0

In [204]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
trend_file_df['LY'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(12)

trend_file_df['LLY'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(24)


trend_file_df['LY value'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(12)

trend_file_df['LLY value'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(24)


trend_file_df['OT_Value_in_Cr_lag_1'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(1)

trend_file_df['OT_Value_in_Cr_lag_2'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(2)

trend_file_df['OT_Value_in_Cr_lag_3'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(3)

In [205]:
for col in ['OT_Value_in_Cr_lag_1', 'OT_Value_in_Cr_lag_2', 'OT_Value_in_Cr_lag_3']:
    # if not 'LY' in col:  'LY P6M',
    trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
    trend_file_df[col] = trend_file_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

In [206]:
trend_file_df.columns

Index(['key', 'month_date', 'pred_SARIMA', 'pred_p3m', 'pred_p6m',
       'pred_prophet', 'pred_rf', 'pred_value_SARIMA', 'pred_value_p3m',
       'pred_value_p6m', 'pred_value_prophet', 'pred_value_rf',
       'parent_material_code', 'platform_name', 'brand_code',
       'great_indian_festival', 'great_indian_festival_lag_1',
       'great_indian_festival_lag_2', 'great_indian_festival_lead_1',
       'great_indian_festival_lead_2', 'qtr_ind_rate', 'vol_in_rum_value',
       'pred_best_model', 'pred_value_best_model', 'vol_in_rum_treated',
       'vol_in_rum_value_treated', 'train_till', 'cov', 'run', 'step',
       'file_path', 'big_billion_days', 'big_billion_days_lag_1',
       'big_billion_days_lag_2', 'big_billion_days_lead_1',
       'big_billion_days_lead_2', 'run_month', 'M month', 'portfolio',
       'pred_prophet_70%ile', 'pred_prophet_60%ile', 'vol_in_rum', 'P3M',
       'P6M', 'LY P3M', 'LY P6M', 'LY P3M_copy', 'P3M Max', 'P3M Top 2 Mean',
       'MoM P3M growth', 'MoM P3M

In [207]:
# trend_file_df[['ASM', 'Depot', 'PSKU']] = trend_file_df['key'].str.split('_', expand=True)

In [208]:
trend_file_df.reset_index(drop=True, inplace=True)

In [209]:
trend_file_df.shape

(111639, 67)

In [210]:
trend_file_df['key'].nunique()

2605

In [211]:
# batch_info = pd.read_excel(
#     '/data/aniket/az_demand_forecasting-mil-sc/channel_wise_batch.xlsx'
# )

In [212]:
# brand_class_df = pd.read_excel(r"/data/aman_singh/acuuracy_check/Brand_Classification.xlsb")
# brand_class_df.head()

In [213]:
# brand_class_df.columns = ['brand_code', 'class']


In [214]:
# len_before_merge = len(trend_file_df)
# trend_file_df = trend_file_df.merge(
#     brand_class_df, 
#     on=['brand_code'],
#     how='left'
# )
# assert len_before_merge == len(trend_file_df)
# del len_before_merge

In [215]:
# trend_file_df['class'].isna().sum()

In [216]:
# trend_file_df['class'].unique()

missing combinations

In [217]:
# model_file = pd.read_csv("/data/aman_singh/acuuracy_check/Heuristic_QCOM_Chain_PSKU_Offtakes_live2.csv")
# model_file

In [218]:
model_file = trend_file_df.copy()

In [219]:
model_file['key'].nunique()

2605

In [220]:
run_month

Timestamp('2026-06-30 00:00:00')

In [221]:
data_query = f"""
    select * from {input_table}
    where month_date >= '2023-01-01' and run_month = '{month_run}'
        
"""
qcom_df = pd.read_sql(data_query, dev_conn)
qcom_df.head()

,MONTH_DATE,PLATFORM_NAME,PARENT_MATERIAL_CODE,BRAND_CODE,VOL_IN_RUM,INDEXBPM,IMPUTED,BIG_BILLION_DAYS,BIG_BILLION_DAYS_LAG_1,BIG_BILLION_DAYS_LAG_2,BIG_BILLION_DAYS_LEAD_1,BIG_BILLION_DAYS_LEAD_2,GREAT_INDIAN_FESTIVAL,GREAT_INDIAN_FESTIVAL_LAG_1,GREAT_INDIAN_FESTIVAL_LAG_2,GREAT_INDIAN_FESTIVAL_LEAD_1,GREAT_INDIAN_FESTIVAL_LEAD_2,RUN_MONTH
0,2023-01-31,Amazon ARIPL,718288,SAFF GOLD,9.640,13.27071,0,0,0,0,0,0,0,0,0,0,0,2026-06-30
1,2023-02-28,Amazon ARIPL,718288,SAFF GOLD,7.915,10.89602,0,0,0,0,0,0,0,0,0,0,0,2026-06-30
2,2023-03-31,Amazon ARIPL,718288,SAFF GOLD,9.505,13.08486,0,0,0,0,0,0,0,0,0,0,0,2026-06-30
3,2023-04-30,Amazon ARIPL,718288,SAFF GOLD,9.290,12.78889,0,0,0,0,0,0,0,0,0,0,0,2026-06-30
4,2023-05-31,Amazon ARIPL,718288,SAFF GOLD,8.050,11.08187,0,0,0,0,0,0,0,0,0,0,0,2026-06-30


In [222]:
qcom_df.columns = qcom_df.columns.str.lower()

In [223]:
qcom_df['key'] = qcom_df[['platform_name', 'parent_material_code']].astype(str).agg('_'.join, axis=1)

In [224]:
model_file['run_month'] = pd.to_datetime(model_file['run_month'])
model_file['month_date'] = pd.to_datetime(model_file['month_date'])

qcom_df['run_month'] = pd.to_datetime(qcom_df['run_month'])
qcom_df['month_date'] = pd.to_datetime(qcom_df['month_date'])

In [225]:
tmp_df = model_file.groupby(['key', 'run_month'])['LY'].count().reset_index()
tmp_df#.isnull().sum()
#qcom_df[~qcom_df['key'].isin(model_file['key'].unique())]
qcom_df = qcom_df.merge(tmp_df, on = ['key', 'run_month'], how = 'left')
missing_df = qcom_df[qcom_df['LY'].isna()]
missing_df


,month_date,platform_name,parent_material_code,brand_code,vol_in_rum,indexbpm,imputed,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,run_month,key,LY
589,2025-08-31,Amazon ARIPL,718494,SFOATS-FL,0.001140,0.00335,0,0,0,0,1,1,0,0,0,1,1,2026-06-30,Amazon ARIPL_718494,NaN
590,2025-09-30,Amazon ARIPL,718494,SFOATS-FL,0.005852,0.01719,0,1,0,0,1,0,1,0,0,1,0,2026-06-30,Amazon ARIPL_718494,NaN
591,2025-10-31,Amazon ARIPL,718494,SFOATS-FL,0.004940,0.01451,0,1,1,0,0,0,1,1,0,0,0,2026-06-30,Amazon ARIPL_718494,NaN
592,2025-11-30,Amazon ARIPL,718494,SFOATS-FL,0.025498,0.07489,0,0,1,1,0,0,0,1,1,0,0,2026-06-30,Amazon ARIPL_718494,NaN
593,2025-12-31,Amazon ARIPL,718494,SFOATS-FL,0.033402,0.09811,0,0,0,1,0,0,0,0,1,0,0,2026-06-30,Amazon ARIPL_718494,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127998,2026-11-30,Nykaa,811019,PADV_WIPS,0.000000,0.00000,1,0,0,0,0,0,0,0,0,0,0,2026-06-30,Nykaa_811019,NaN
127999,2026-12-31,Nykaa,811019,PADV_WIPS,0.000000,0.00000,1,0,0,0,0,0,0,0,0,0,0,2026-06-30,Nykaa_811019,NaN
128000,2027-01-31,Nykaa,811019,PADV_WIPS,0.000000,0.00000,1,0,0,0,0,0,0,0,0,0,0,2026-06-30,Nykaa_811019,NaN
128001,2027-02-28,Nykaa,811019,PADV_WIPS,0.000000,0.00000,1,0,0,0,0,0,0,0,0,0,0,2026-06-30,Nykaa_811019,NaN


In [226]:
missing_df['key'].nunique()

618

In [227]:
missing_df = missing_df[['key','run_month','month_date', 'platform_name', 'parent_material_code', 'brand_code',
       'vol_in_rum']]
missing_df

,key,run_month,month_date,platform_name,parent_material_code,brand_code,vol_in_rum
589,Amazon ARIPL_718494,2026-06-30,2025-08-31,Amazon ARIPL,718494,SFOATS-FL,0.001140
590,Amazon ARIPL_718494,2026-06-30,2025-09-30,Amazon ARIPL,718494,SFOATS-FL,0.005852
591,Amazon ARIPL_718494,2026-06-30,2025-10-31,Amazon ARIPL,718494,SFOATS-FL,0.004940
592,Amazon ARIPL_718494,2026-06-30,2025-11-30,Amazon ARIPL,718494,SFOATS-FL,0.025498
593,Amazon ARIPL_718494,2026-06-30,2025-12-31,Amazon ARIPL,718494,SFOATS-FL,0.033402
...,...,...,...,...,...,...,...
127998,Nykaa_811019,2026-06-30,2026-11-30,Nykaa,811019,PADV_WIPS,0.000000
127999,Nykaa_811019,2026-06-30,2026-12-31,Nykaa,811019,PADV_WIPS,0.000000
128000,Nykaa_811019,2026-06-30,2027-01-31,Nykaa,811019,PADV_WIPS,0.000000
128001,Nykaa_811019,2026-06-30,2027-02-28,Nykaa,811019,PADV_WIPS,0.000000


In [228]:
qtr_df = read_qtr_ind_rate_table()
qtr_df.columns = qtr_df.columns.str.lower()
qtr_df.head()


Credentials retrieved successfully for prod db.


,month_date,brand_code,qtr_ind_rate
0,2027-03-31,PA_CN_HGO,488.152
1,2027-03-31,TRU_RAWDF,800.000
2,2027-03-31,TRU_PDRFR,850.570
3,2027-03-31,TRU_OATS,177.070
4,2027-03-31,TRU_QUINO,204.750


In [229]:
len_before_merge = len(missing_df)

missing_df = missing_df.rename(columns={'material_group_code': 'brand_code'}).merge(
    qtr_df.drop('month_date', axis=1),
    on=['brand_code'],
    how='left'
)

assert len_before_merge == len(missing_df)

In [230]:
missing_df

,key,run_month,month_date,platform_name,parent_material_code,brand_code,vol_in_rum,qtr_ind_rate
0,Amazon ARIPL_718494,2026-06-30,2025-08-31,Amazon ARIPL,718494,SFOATS-FL,0.001140,292663.137458
1,Amazon ARIPL_718494,2026-06-30,2025-09-30,Amazon ARIPL,718494,SFOATS-FL,0.005852,292663.137458
2,Amazon ARIPL_718494,2026-06-30,2025-10-31,Amazon ARIPL,718494,SFOATS-FL,0.004940,292663.137458
3,Amazon ARIPL_718494,2026-06-30,2025-11-30,Amazon ARIPL,718494,SFOATS-FL,0.025498,292663.137458
4,Amazon ARIPL_718494,2026-06-30,2025-12-31,Amazon ARIPL,718494,SFOATS-FL,0.033402,292663.137458
...,...,...,...,...,...,...,...,...
11149,Nykaa_811019,2026-06-30,2026-11-30,Nykaa,811019,PADV_WIPS,0.000000,366.484998
11150,Nykaa_811019,2026-06-30,2026-12-31,Nykaa,811019,PADV_WIPS,0.000000,366.484998
11151,Nykaa_811019,2026-06-30,2027-01-31,Nykaa,811019,PADV_WIPS,0.000000,366.484998
11152,Nykaa_811019,2026-06-30,2027-02-28,Nykaa,811019,PADV_WIPS,0.000000,366.484998


In [231]:
missing_df['month_date'] = pd.to_datetime(missing_df['month_date'])
missing_df['run_month'] = pd.to_datetime(missing_df['run_month'])


mappings = {}

for run_month in missing_df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 11):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   
missing_df['M month'] = missing_df.apply(
    lambda x: mappings[x['run_month']].get(
        x['month_date']
    ), axis=1
)

brand_md_df = pd.read_excel(r"/data/aman_singh/mt_forecast/Brand_metadata.xlsx")
len_before_merge = len(missing_df)
missing_df = missing_df.merge(
    brand_md_df,
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(missing_df)

# Merge 70th percentile Prophet predictions

assert missing_df.duplicated(
    subset=['run_month', 'month_date', 'key']
).sum() == 0
missing_df.drop('vol_in_rum', axis=1, inplace=True)

In [232]:
missing_df['M month'].unique()

array([None, 'M', 'M+1', 'M+2', 'M+3', 'M+4', 'M+5', 'M+6', 'M+7', 'M+8',
       'M+9'], dtype=object)

In [233]:
offtake_df['run_month'] = pd.to_datetime(offtake_df['run_month'])
offtake_df['month_date'] = pd.to_datetime(offtake_df['month_date'])
assert offtake_df.duplicated(subset=['key', 'month_date']).sum() == 0
len_before_merge = len(missing_df)
missing_df = missing_df.merge(
    offtake_df[['key', 'month_date', 'vol_in_rum']],
    on=['month_date', 'key'],
    how='left'
)
assert len(missing_df) == len_before_merge

missing_df['vol_in_rum'].fillna(0, inplace=True)
for col in [ 'vol_in_rum']:
    missing_df[col] = missing_df[col].clip(lower=0)
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)

In [234]:
missing_df['P3M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=3, min_periods=1).mean()

missing_df['P6M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=6, min_periods=3).mean()

missing_df['LY P3M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=3, min_periods=3).mean()

missing_df['LY P6M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=6, min_periods=6).mean()

In [235]:

missing_df['LY P3M_copy'] = missing_df['LY P3M'].copy()
missing_df['P3M Max'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).max()
missing_df['P3M Top 2 Mean'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sort(x)[-2:].mean(), raw=True) 

# lambda x: x.nlargest(2).mean(), raw=False
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
missing_df['MoM P3M growth'] = (
    missing_df.groupby(['run_month', 'key'])['P3M']
      .pct_change() * 100
)
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
missing_df['MoM P3M growth_lag_1'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(1)
missing_df['MoM P3M growth_lag_2'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(2)


missing_df['>=20%_3M_inc_month_count'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['MoM P3M growth'].shift(0)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sum(x >= 20), raw=True) 
missing_df['Avg(P3M Mean, Max)'] = missing_df[['P3M', 'P3M Max']].mean(axis=1)
for col in ['P3M', 'P6M', 'LY P3M', 'P3M Max', 'P3M Top 2 Mean', 
            'MoM P3M growth', '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)',
            'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2' ]:
    # if not 'LY' in col:  'LY P6M',
    missing_df.loc[missing_df['month_date'] > missing_df['run_month'], [col]] = np.nan
    missing_df[col] = missing_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )


# for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
#     if not 'LY' in col:
#         missing_df.loc[missing_df['month_date'] > missing_df['run_month'], [col]] = np.nan
#         missing_df[col] = missing_df.groupby(['key'], as_index = True, group_keys = False)[col].apply(
#             lambda x: x.ffill()
#         )
# missing_df.to_csv('collate_check.csv', index=False)
for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
    missing_df[f'{col}_value'] = missing_df[col] * missing_df['qtr_ind_rate'] / (10 ** 7)
missing_df['vol_in_rum_value'] = missing_df['qtr_ind_rate'] * missing_df['vol_in_rum'] / (10 ** 7)
value_cols = [col for col in missing_df.columns if 'value' in col]
value_cols
for col in value_cols:
    try:
        assert missing_df[col].min() >= 0
    except:
        print(col)
    

    # missing_df[col] = missing_df[col] / (10 ** 7)
missing_df
assert missing_df.duplicated(
    subset=['run_month', 'brand_code', 'key', 'month_date']
).sum() == 0
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
missing_df['LY'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(12)

missing_df['LLY'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(24)


missing_df['LY value'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(12)

missing_df['LLY value'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(24)


missing_df['OT_Value_in_Cr_lag_1'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(1)

missing_df['OT_Value_in_Cr_lag_2'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(2)

missing_df['OT_Value_in_Cr_lag_3'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(3)
for col in ['OT_Value_in_Cr_lag_1', 'OT_Value_in_Cr_lag_2', 'OT_Value_in_Cr_lag_3']:
    # if not 'LY' in col:  'LY P6M',
    missing_df.loc[missing_df['month_date'] > missing_df['run_month'], [col]] = np.nan
    missing_df[col] = missing_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

missing_df.reset_index(drop=True, inplace=True)

# brand_class_df = pd.read_excel(r"/data/aman_singh/acuuracy_check/Brand_Classification.xlsb")
# brand_class_df.head()
# brand_class_df.columns = ['brand_code', 'class']

# len_before_merge = len(missing_df)
# missing_df = missing_df.merge(
#     brand_class_df, 
#     on=['brand_code'],
#     how='left'
# )
# assert len_before_merge == len(missing_df)
# del len_before_merge
# missing_df['class'].isna().sum()
# missing_df['class'].unique()

LY P3M_value


In [236]:
pd.set_option('display.max_columns', None)

In [237]:
# missing_df[
#     # (missing_df['channel'].isin(['MT', 'QCOM'])) & 
#     # (missing_df['month_date'] > '2024-06-30') &
#     (missing_df['M month'].notna())
#     # (missing_df['class'].isin(['B', 'C']))
# ].to_csv('missing_combinations_QCOM_Chain_city_PSKU_Offtakes.csv', index=False)

In [238]:
model_file

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3
0,Amazon RK_718472,2023-11-30,0.000000,0.51,1.125,0.452544,0.765,0.000000,0.000025,0.000056,0.000022,0.000038,718472,Amazon RK,ADV-AHO-R,0.0,1.0,0.0,0.0,0.0,496.828458,0.000022,NaN,NaN,0.45,0.000022,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Hair Oils,0.676890,0.547432,0.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000034,0.000027,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Amazon RK_718472,2023-12-31,0.450000,0.51,1.125,0.289509,0.774,0.000022,0.000025,0.000056,0.000014,0.000038,718472,Amazon RK,ADV-AHO-R,0.0,0.0,1.0,0.0,0.0,496.828458,0.000000,NaN,NaN,0.00,0.000000,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Hair Oils,0.545380,0.432579,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000027,0.000021,NaN,NaN,NaN,NaN,0.000022,NaN,NaN
2,Amazon RK_718472,2024-01-31,0.172500,0.51,1.125,1.143780,1.116,0.000009,0.000025,0.000056,0.000057,0.000055,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000054,NaN,NaN,1.08,0.000054,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Hair Oils,1.380028,1.239281,1.08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000069,0.000062,NaN,NaN,NaN,NaN,0.000000,0.000022,NaN
3,Amazon RK_718472,2024-02-29,0.678074,0.51,1.125,1.165967,1.296,0.000034,0.000025,0.000056,0.000058,0.000064,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000080,NaN,NaN,1.62,0.000080,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Hair Oils,1.416077,1.293417,1.62,0.51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.51,0.000025,NaN,NaN,NaN,0.000070,0.000064,NaN,NaN,NaN,NaN,0.000054,0.000000,0.000022
4,Amazon RK_718472,2024-03-31,0.844506,0.90,1.125,1.783082,1.818,0.000042,0.000045,0.000056,0.000089,0.000090,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000134,NaN,NaN,2.70,0.000134,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Hair Oils,2.008802,1.887959,2.70,0.90,NaN,NaN,NaN,NaN,NaN,NaN,76.470588,NaN,NaN,NaN,0.90,0.000045,NaN,NaN,NaN,0.000100,0.000094,NaN,NaN,NaN,NaN,0.000080,0.000054,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111634,Big Basket_719192,2026-09-30,0.000000,0.00,0.000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,719192,Big Basket,VEG_CLEAN,NaN,NaN,NaN,NaN,NaN,100.000000,0.000000,NaN,NaN,0.00,0.000000,2026-05-31,4.605489,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,

In [239]:
model_file['skipped'] = 0
missing_df['skipped'] = 1
final_df = pd.concat([model_file,missing_df])
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped
0,Amazon RK_718472,2023-11-30,0.000000,0.51,1.125,0.452544,0.765,0.000000,0.000025,0.000056,0.000022,0.000038,718472,Amazon RK,ADV-AHO-R,0.0,1.0,0.0,0.0,0.0,496.828458,0.000022,NaN,NaN,0.45,0.000022,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Hair Oils,0.676890,0.547432,0.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000034,0.000027,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,Amazon RK_718472,2023-12-31,0.450000,0.51,1.125,0.289509,0.774,0.000022,0.000025,0.000056,0.000014,0.000038,718472,Amazon RK,ADV-AHO-R,0.0,0.0,1.0,0.0,0.0,496.828458,0.000000,NaN,NaN,0.00,0.000000,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Hair Oils,0.545380,0.432579,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000027,0.000021,NaN,NaN,NaN,NaN,0.000022,NaN,NaN,0
2,Amazon RK_718472,2024-01-31,0.172500,0.51,1.125,1.143780,1.116,0.000009,0.000025,0.000056,0.000057,0.000055,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000054,NaN,NaN,1.08,0.000054,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Hair Oils,1.380028,1.239281,1.08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000069,0.000062,NaN,NaN,NaN,NaN,0.000000,0.000022,NaN,0
3,Amazon RK_718472,2024-02-29,0.678074,0.51,1.125,1.165967,1.296,0.000034,0.000025,0.000056,0.000058,0.000064,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000080,NaN,NaN,1.62,0.000080,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Hair Oils,1.416077,1.293417,1.62,0.510,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.510,0.000025,NaN,NaN,NaN,0.000070,0.000064,NaN,NaN,NaN,NaN,0.000054,0.000000,0.000022,0
4,Amazon RK_718472,2024-03-31,0.844506,0.90,1.125,1.783082,1.818,0.000042,0.000045,0.000056,0.000089,0.000090,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000134,NaN,NaN,2.70,0.000134,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Hair Oils,2.008802,1.887959,2.70,0.900,NaN,NaN,NaN,NaN,NaN,NaN,76.470588,NaN,NaN,NaN,0.900,0.000045,NaN,NaN,NaN,0.000100,0.000094,NaN,NaN,NaN,NaN,0.000080,0.000054,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11149,Myntra_811169,2026-11-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,811169,Myntra,SW_SGPRF,NaN,NaN,NaN,NaN,NaN,1712.605337,0.000000,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-30,M+5,Male Grooming,NaN,NaN,0.00,2.304,0.9216,NaN,NaN,NaN,NaN,NaN,inf,NaN,NaN,NaN,2.304,0

In [240]:
final_df[final_df.select_dtypes(include='number').columns] = final_df.select_dtypes(include='number').fillna(0)
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped
0,Amazon RK_718472,2023-11-30,0.000000,0.51,1.125,0.452544,0.765,0.000000,0.000025,0.000056,0.000022,0.000038,718472,Amazon RK,ADV-AHO-R,0.0,1.0,0.0,0.0,0.0,496.828458,0.000022,0.0,0.0,0.45,0.000022,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,0.676890,0.547432,0.45,0.000,0.0000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000,0.000000,0.000000,0.0,0.0,0.000034,0.000027,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0
1,Amazon RK_718472,2023-12-31,0.450000,0.51,1.125,0.289509,0.774,0.000022,0.000025,0.000056,0.000014,0.000038,718472,Amazon RK,ADV-AHO-R,0.0,0.0,1.0,0.0,0.0,496.828458,0.000000,0.0,0.0,0.00,0.000000,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,0.545380,0.432579,0.00,0.000,0.0000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000,0.000000,0.000000,0.0,0.0,0.000027,0.000021,0.0,0.0,0.0,0.0,0.000022,0.000000,0.000000,0
2,Amazon RK_718472,2024-01-31,0.172500,0.51,1.125,1.143780,1.116,0.000009,0.000025,0.000056,0.000057,0.000055,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000054,0.0,0.0,1.08,0.000054,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,1.380028,1.239281,1.08,0.000,0.0000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000,0.000000,0.000000,0.0,0.0,0.000069,0.000062,0.0,0.0,0.0,0.0,0.000000,0.000022,0.000000,0
3,Amazon RK_718472,2024-02-29,0.678074,0.51,1.125,1.165967,1.296,0.000034,0.000025,0.000056,0.000058,0.000064,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000080,0.0,0.0,1.62,0.000080,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,1.416077,1.293417,1.62,0.510,0.0000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.510,0.000025,0.000000,0.0,0.0,0.000070,0.000064,0.0,0.0,0.0,0.0,0.000054,0.000000,0.000022,0
4,Amazon RK_718472,2024-03-31,0.844506,0.90,1.125,1.783082,1.818,0.000042,0.000045,0.000056,0.000089,0.000090,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000134,0.0,0.0,2.70,0.000134,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,2.008802,1.887959,2.70,0.900,0.0000,0.0,0.0,0.0,0.0,0.0,76.470588,0.0,0.0,0.0,0.900,0.000045,0.000000,0.0,0.0,0.000100,0.000094,0.0,0.0,0.0,0.0,0.000080,0.000054,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11149,Myntra_811169,2026-11-30,0.000000,0.00,0.000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,811169,Myntra,SW_SGPRF,0.0,0.0,0.0,0.0,0.0,1712.605337,0.0000

In [241]:
final_df['month_date'] = pd.to_datetime(final_df['month_date'])
final_df['run_month'] = pd.to_datetime(final_df['run_month'])
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped
0,Amazon RK_718472,2023-11-30,0.000000,0.51,1.125,0.452544,0.765,0.000000,0.000025,0.000056,0.000022,0.000038,718472,Amazon RK,ADV-AHO-R,0.0,1.0,0.0,0.0,0.0,496.828458,0.000022,0.0,0.0,0.45,0.000022,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,0.676890,0.547432,0.45,0.000,0.0000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000,0.000000,0.000000,0.0,0.0,0.000034,0.000027,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0
1,Amazon RK_718472,2023-12-31,0.450000,0.51,1.125,0.289509,0.774,0.000022,0.000025,0.000056,0.000014,0.000038,718472,Amazon RK,ADV-AHO-R,0.0,0.0,1.0,0.0,0.0,496.828458,0.000000,0.0,0.0,0.00,0.000000,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,0.545380,0.432579,0.00,0.000,0.0000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000,0.000000,0.000000,0.0,0.0,0.000027,0.000021,0.0,0.0,0.0,0.0,0.000022,0.000000,0.000000,0
2,Amazon RK_718472,2024-01-31,0.172500,0.51,1.125,1.143780,1.116,0.000009,0.000025,0.000056,0.000057,0.000055,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000054,0.0,0.0,1.08,0.000054,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,1.380028,1.239281,1.08,0.000,0.0000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000,0.000000,0.000000,0.0,0.0,0.000069,0.000062,0.0,0.0,0.0,0.0,0.000000,0.000022,0.000000,0
3,Amazon RK_718472,2024-02-29,0.678074,0.51,1.125,1.165967,1.296,0.000034,0.000025,0.000056,0.000058,0.000064,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000080,0.0,0.0,1.62,0.000080,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,1.416077,1.293417,1.62,0.510,0.0000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.510,0.000025,0.000000,0.0,0.0,0.000070,0.000064,0.0,0.0,0.0,0.0,0.000054,0.000000,0.000022,0
4,Amazon RK_718472,2024-03-31,0.844506,0.90,1.125,1.783082,1.818,0.000042,0.000045,0.000056,0.000089,0.000090,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000134,0.0,0.0,2.70,0.000134,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,2.008802,1.887959,2.70,0.900,0.0000,0.0,0.0,0.0,0.0,0.0,76.470588,0.0,0.0,0.0,0.900,0.000045,0.000000,0.0,0.0,0.000100,0.000094,0.0,0.0,0.0,0.0,0.000080,0.000054,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11149,Myntra_811169,2026-11-30,0.000000,0.00,0.000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,811169,Myntra,SW_SGPRF,0.0,0.0,0.0,0.0,0.0,1712.605337,0.0000

Heuristic new approac

In [242]:
# pip install pymannkendall

identify events

In [243]:
final_df['month_date'] = pd.to_datetime(final_df['month_date'])
final_df['run_month'] = pd.to_datetime(final_df['run_month'])

In [244]:
import numpy as np 
import pandas as pd 
EVENT_MONTHS = [9, 10, 11] # Sep, Oct, Nov
def detect_event_months(df_grp):
    df_grp = df_grp.sort_values("month_date").copy()
    run_month = df_grp["run_month"].max()

    # only historical data
    hist = df_grp[df_grp["month_date"] < run_month].copy()

    # initialize
    df_grp["event_month_flag"] = 0
    df_grp["event_uplift_factor"] = 0.0

    if len(hist) < 12:
        df_grp["event_sensitive_flag"] = 0
        return df_grp

    event_sensitive = 0

    # loop year-wise
    for year in hist["month_date"].dt.year.unique():

        year_df = hist[hist["month_date"].dt.year == year]

        for _, row in year_df.iterrows():

            month = row["month_date"].month

            if month not in EVENT_MONTHS:
                continue

            curr_date = row["month_date"]

            # previous 12 months before this month
            prev_12m = hist[
                (hist["month_date"] < curr_date) &
                (hist["month_date"] >= curr_date - pd.DateOffset(months=12)) &
                (~hist["month_date"].dt.month.isin(EVENT_MONTHS))  # 
            ]

            if len(prev_12m) < 6:
                continue

            prev_12m_avg = prev_12m["vol_in_rum_value"].mean()

            if prev_12m_avg <= 0 or np.isnan(prev_12m_avg):
                continue

            uplift = row["vol_in_rum_value"] / prev_12m_avg

            if uplift > 2:
                mask = df_grp["month_date"] == curr_date
                df_grp.loc[mask, "event_month_flag"] = 1
                df_grp.loc[mask, "event_uplift_factor"] = uplift
                event_sensitive = 1

    df_grp["event_sensitive_flag"] = event_sensitive
    return df_grp


In [245]:
final_df = (
    final_df
    .groupby(
        ["platform_name", "parent_material_code", "run_month"],
        group_keys=False
    )
    .apply(detect_event_months)
)


In [246]:
final_df[(final_df['event_month_flag'] == 1) & (final_df['platform_name'] == 'Flipkart National') & ((final_df['parent_material_code'] == 718939))]

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag
73382,Flipkart National_718939,2023-09-30,116.087138,111.709333,86.266000,186.251111,178.193898,1.364531,1.313073,1.014003,2.189265,2.094557,718939,Flipkart National,SAFF ACTV,0.0,0.0,0.0,0.0,0.0,117543.724493,2.186454,0.0,0.0,186.012000,2.186454,2026-05-31,0.767965,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,1.0,0.0,2026-06-30,None,Saffola Oils,198.590465,192.315576,186.012000,111.709333,86.266000,0.000000,0.000000,0.000000,99.329333,90.655333,12.463589,21.160915,34.787470,2.0,105.519333,1.313073,1.014003,0.000000,0.000000,2.334306,2.260549,0.000,0.000,0.000000,0.000000,1.593188,0.996912,1.349120,0,1,2.251920,1
73383,Flipkart National_718939,2023-10-31,116.087138,135.454667,108.718000,365.569821,277.458849,1.364531,1.592185,1.277912,4.297044,3.261355,718939,Flipkart National,SAFF ACTV,0.0,0.0,0.0,0.0,0.0,117543.724493,4.293919,0.0,0.0,365.304000,4.293919,2026-05-31,0.767965,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,1.0,0.0,0.0,0.0,0.0,2026-06-30,None,Saffola Oils,375.369904,369.773097,365.304000,135.454667,108.718000,0.000000,0.000000,0.000000,111.709333,105.519333,21.256356,12.463589,21.160915,2.0,123.582000,1.592185,1.277912,0.000000,0.000000,4.412238,4.346451,0.000,0.000,0.000000,0.000000,2.186454,1.593188,0.996912,0,1,4.422486,1
73384,Flipkart National_718939,2023-11-30,116.087138,228.952000,164.140667,189.071118,158.175733,1.364531,2.691187,1.929371,2.222412,1.859256,718939,Flipkart National,SAFF ACTV,0.0,0.0,0.0,0.0,0.0,117543.724493,2.220166,0.0,0.0,188.880000,2.220166,2026-05-31,0.767965,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,1.0,0.0,0.0,0.0,2026-06-30,None,Saffola Oils,205.074431,198.152653,188.880000,228.952000,164.140667,0.000000,0.000000,0.000000,135.454667,123.582000,69.024815,21.256356,12.463589,2.0,182.203333,2.691187,1.929371,0.000000,0.000000,2.410521,2.329160,0.000,0.000,0.000000,0.000000,4.293919,2.186454,1.593188,0,1,2.286641,1
73394,Flipkart National_718939,2024-09-30,165.671057,115.513333,84.795333,285.419100,309.434749,1.947359,1.357787,0.996716,3.354922,3.637211,718939,Flipkart National,SAFF ACTV,0.0,0.0,0.0,0.0,0.0,117543.724493,3.490061,0.0,0.0,296.916000,3.490061,2026-05-31,0.767965,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,1.0,0.0,0.0,1.0,0.0,2026-06-30,None,Saffola Oils,297.765162,291.898025,296.916000,115.513333,84.795333,111.709333,86.266000,111.709333,99.206667,85.028667,16.437067,40.022206,31.017309,2.0,107.360000,1.357787,0.996716,1.313073,1.014003,3.500043,3.431078,186.012,0.000,2.186454,0.000000,2.056310,1.234820,0.782230,0,1,3.377611,1
73395,Flipkart National_718939,2024-10-31,292.807525,192.302667,131.576667,240.482636,244.068539,3.441769,2.260397,1.546601,2.826722,2.868873,718939,Flipkart National,SAFF ACTV,0.0,0.0,0.0,0.0,0.0,117543.724493,2.815877,0.0,0.0,239.560000,2.815877,2026-05-31,0.767965,run,acuura

In [247]:
def compute_adjusted_pm(df_grp, window, column, year_shift=0):
    df_grp = df_grp.sort_values("month_date")

    run_month = df_grp["run_month"].iloc[0]

    # define cutoff
    end_date = run_month - pd.DateOffset(years=year_shift)

    # keep only eligible history (before run month & non-event)
    hist = df_grp[
        (df_grp["month_date"] < end_date) &
        (df_grp["event_month_flag"] == 0)
    ]

    if hist.empty:
        return np.nan

    # take last `window` non-event months
    hist = hist.tail(window)

    # if len(hist) < window:
    #     return np.nan   # optional, keeps behavior strict

    return hist[column].mean()


In [248]:
adj_df = final_df.groupby(
    ["platform_name", "parent_material_code", "run_month"]
).apply(
    lambda x: pd.Series({
        "P3M_adj": compute_adjusted_pm(x, 3,'vol_in_rum',0),
        "P6M_adj": compute_adjusted_pm(x, 6,'vol_in_rum',0),
        "P3M_adj_value": compute_adjusted_pm(x, 3,'vol_in_rum_value',0),
        "P6M_adj_value": compute_adjusted_pm(x, 6,'vol_in_rum_value',0)
    })
).reset_index()

final_df = final_df.merge(
    adj_df,
    on=["platform_name", "parent_material_code", "run_month"],
    how="left"
)

In [249]:
adj_ly_df = final_df.groupby(
    ["platform_name", "parent_material_code", "run_month"]
).apply(
    lambda x: pd.Series({
        "LY_P3M_adj": compute_adjusted_pm(x, 3,'vol_in_rum',year_shift=1),
        "LY_P6M_adj": compute_adjusted_pm(x, 6,'vol_in_rum', year_shift=1),
        "LY_P3M_adj_value": compute_adjusted_pm(x, 3,'vol_in_rum_value', year_shift=1),
        "LY_P6M_adj_value": compute_adjusted_pm(x, 6,"vol_in_rum_value", year_shift=1)
    })
).reset_index()

In [250]:
final_df = final_df.merge(
    adj_ly_df,
    on=["platform_name", "parent_material_code", "run_month"],
    how="left"
)
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value
0,Amazon RK_718472,2023-11-30,0.0000,0.51,1.125,0.452544,0.765,0.000000,0.000025,0.000056,0.000022,0.000038,718472,Amazon RK,ADV-AHO-R,0.0,1.0,0.0,0.0,0.0,496.828458,0.000022,0.0,0.0,0.45,0.000022,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,0.676890,0.547432,0.45,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000034,0.000027,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.00000,0.0,0.0,0.0,0.0
1,Meesho_718472,2025-12-31,0.0000,0.00,0.000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.002119,0.0,0.0,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,0.000000,0.000000,42.66,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1,0,0.0,0,1.26,9.45,0.000063,0.00047,NaN,NaN,NaN,NaN
2,Amazon RK_718472,2023-12-31,0.4500,0.51,1.125,0.289509,0.774,0.000022,0.000025,0.000056,0.000014,0.000038,718472,Amazon RK,ADV-AHO-R,0.0,0.0,1.0,0.0,0.0,496.828458,0.000000,0.0,0.0,0.00,0.000000,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,0.545380,0.432579,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000027,0.000021,0.0,0.0,0.0,0.0,0.000022,0.000000,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.00000,0.0,0.0,0.0,0.0
3,Meesho_718472,2026-01-31,0.0000,0.00,0.000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000456,0.0,0.0,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,0.000000,0.000000,9.18,42.66,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,42.66,0.002119,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.002119,0.000000,0.0,1,0,0.0,0,1.26,9.45,0.000063,0.00047,NaN,NaN,NaN,NaN
4,Amazon RK_718472,2024-01-31,0.1725,0.51,1.125,1.143780,1.116,0.000009,0.000025,0.000056,0.000057,0.000055,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000054,0.0,0.0,1.08,0.000054,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,1.380028,1.239281,1.08,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000069,0.000062,0.0,0.0,0.0,0.0,0.000000,0.000022,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.00000,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...

In [251]:
qtr_df = read_qtr_ind_rate_table()
qtr_df.columns = qtr_df.columns.str.lower()
qtr_df.head()


len_before_merge = len(final_df)

final_df = final_df.merge(
    qtr_df.drop('month_date', axis=1),
    on=['brand_code'],
    how='left'
)

assert len_before_merge == len(final_df)


Credentials retrieved successfully for prod db.


In [252]:
import pandas as pd
import numpy as np
import pymannkendall as mk

def detect_trend_for_group(df_grp):
    """
    Detect final trend flag and p3m_slope_flag separately.
    Must contain 'month_date', 'vol_in_rum', 'run_month'
    """

    # ---------- 1. Sort ----------
    df_grp = df_grp.sort_values("month_date")

    # ---------- 2. Identify run_month ----------
    run_month = df_grp["run_month"].max()

    # actual data = months < run_month
    df_actual = df_grp[df_grp["month_date"] < run_month]

    # if no actual data → no trend
    if df_actual.empty or len(df_actual) < 4:
        return pd.Series({"trend_flag": 0, "p3m_slope_flag": 0})

    # ---------- 3. MK Trend ----------
    series = df_actual["vol_in_rum_value"].astype(float)

    try:
        mk_result = mk.original_test(series)
        if mk_result.trend == "increasing":
            mk_trend = 1
        elif mk_result.trend == "decreasing":
            mk_trend = -1
        else:
            mk_trend = 0
    except:
        mk_trend = 0

    # ---------- 4. P3M Slope ----------
    # last 4 months → take last 3 with shift
    #shifted_series = series.shift(1).dropna()

    p3m_values = series.tail(3).values
    #print(p3m_values)

    if len(p3m_values) < 3:
        slope_flag = 0
    else:
        x = np.arange(3)
        slope = np.polyfit(x, p3m_values, 1)[0]
        slope_flag = 1 if slope > 0 else (-1 if slope < 0 else 0)
        

    return pd.Series({
        "trend_flag": mk_trend,
        "p3m_slope_flag": slope_flag
    })


# ---------------------------------------------------------
# APPLY ON ENTIRE DATASET
# ---------------------------------------------------------

# trend_df = final_df.groupby(
#     ["platform_name", "parent_material_code"]
# ).apply(detect_trend_for_group).reset_index()

# trend_df = final_df[final_df['key'] == 'Zepto_721898'].groupby(
#     ["platform_name", "parent_material_code", "run_month"]
# ).apply(detect_trend_for_group).reset_index()
trend_df = final_df.groupby(
    ["platform_name", "parent_material_code", "run_month"]
).apply(detect_trend_for_group).reset_index()

In [253]:
trend_df["final_trend"] = np.where(
    (trend_df["trend_flag"] == 1) & (trend_df["p3m_slope_flag"] == 1), 1,
    np.where(
        (trend_df["trend_flag"] == -1) & (trend_df["p3m_slope_flag"] == -1), -1,
        0
    )
)
trend_df

,platform_name,parent_material_code,run_month,trend_flag,p3m_slope_flag,final_trend
0,Amazon ARIPL,718288,2026-06-30,1,-1,0
1,Amazon ARIPL,718321,2026-06-30,-1,0,0
2,Amazon ARIPL,718322,2026-06-30,0,-1,0
3,Amazon ARIPL,718323,2026-06-30,-1,0,0
4,Amazon ARIPL,718328,2026-06-30,1,-1,0
...,...,...,...,...,...,...
3218,Nykaa,810673,2026-06-30,0,1,0
3219,Nykaa,810674,2026-06-30,0,1,0
3220,Nykaa,810738,2026-06-30,-1,0,0
3221,Nykaa,810805,2026-06-30,0,-1,0


## detect seasonality

In [254]:
from statsmodels.tsa.stattools import acf
import numpy as np
import pandas as pd

def detect_yearly_seasonality(df_grp, threshold=0.3):
    """
    Detects yearly seasonality using ACF at lag=12 only.
    Uses vol_in_rum as the metric.
    """
    df_grp = df_grp.sort_values("month_date")
    run_month = df_grp["run_month"].max()

    # actual data = months < run_month
    df_actual = df_grp[df_grp["month_date"] < run_month]
    series = df_actual["vol_in_rum"].astype(float).values

    # Need at least 18 points to compare last year vs this year
    if len(series) < 18:
        return 0

    # Compute ACF up to lag-12
    acf_vals = acf(series, nlags=12, fft=False)

    lag12_acf = acf_vals[12]

    # absolute ACF because seasonal correlation can be negative as well
    if abs(lag12_acf) >= threshold:
        return 1
    else:
        return 0
    

seasonality_df = final_df.groupby(
    ["platform_name", "brand_code", 'run_month']
).apply(detect_yearly_seasonality).reset_index(name="seasonality_flag")

seasonality_df



,platform_name,brand_code,run_month,seasonality_flag
0,Amazon ARIPL,CO_SO_VCN,2026-06-30,0
1,Amazon ARIPL,SAF-MUSLI,2026-06-30,0
2,Amazon ARIPL,SAFF ACTV,2026-06-30,0
3,Amazon ARIPL,SAFF GOLD,2026-06-30,0
4,Amazon ARIPL,SAFF KO,2026-06-30,0
...,...,...,...,...
580,Nykaa,SW HRGEL,2026-06-30,0
581,Nykaa,SW NOGAS,2026-06-30,1
582,Nykaa,SW STLDEO,2026-06-30,0
583,Nykaa,SW_HR_WAX,2026-06-30,0


In [255]:
# seasonality_df.to_csv('seasonal_ecom.csv')

In [256]:
import numpy as np
import pandas as pd

def compute_thresholds(df_grp):
    """
    df_grp MUST contain:
    - month_date
    - vol_in_rum
    - run_month

    Returns: lower_threshold, upper_threshold, mean, std
    """

    df_grp = df_grp.sort_values("month_date")
    run_month = df_grp["run_month"].max()

    # --- Use ONLY actual data (strictly before run month)
    df_actual = df_grp[df_grp["month_date"] < run_month]

    series = df_actual["vol_in_rum_value"].astype(float).values

    # If no real data → return zeros
    if len(series) == 0:
        return pd.Series({
            "lower_threshold": 0,
            "upper_threshold": 0,
            "mean_value": 0,
            "std_value": 0
        })

    # --- Take last 12 months OR all available
    if len(series) > 12:
        series = series[-12:]

    mean_val = np.mean(series)
    std_val = np.std(series)

    # --- SPECIAL CASE: ≤3 data points
    if len(series) <= 3:
        lower = 0.5 * mean_val
        upper = 2 * mean_val

        return pd.Series({
            "lower_threshold": lower,
            "upper_threshold": upper,
            "mean_value": mean_val,
            "std_value": std_val
        })

    # --- Normal case (std can be zero also)
    lower = max(0,mean_val - 2*std_val)
    upper = mean_val + 3*std_val

    return pd.Series({
        "lower_threshold": lower,
        "upper_threshold": upper,
        "mean_value": mean_val,
        "std_value": std_val
    })

threshold_df = final_df.groupby(
    ["platform_name", "parent_material_code", "run_month"]
).apply(compute_thresholds).reset_index()

threshold_df.head()


,platform_name,parent_material_code,run_month,lower_threshold,upper_threshold,mean_value,std_value
0,Amazon ARIPL,718288,2026-06-30,0.104661,0.587795,0.297915,0.096627
1,Amazon ARIPL,718321,2026-06-30,0.000000,0.000000,0.000000,0.000000
2,Amazon ARIPL,718322,2026-06-30,0.000000,0.419733,0.141604,0.092710
3,Amazon ARIPL,718323,2026-06-30,0.000000,0.000000,0.000000,0.000000
4,Amazon ARIPL,718328,2026-06-30,0.000000,0.173154,0.055332,0.039274


In [257]:
trend_df = trend_df.merge(threshold_df, on = ['platform_name', 'parent_material_code', 'run_month'], how = 'left')
trend_df

,platform_name,parent_material_code,run_month,trend_flag,p3m_slope_flag,final_trend,lower_threshold,upper_threshold,mean_value,std_value
0,Amazon ARIPL,718288,2026-06-30,1,-1,0,0.104661,0.587795,0.297915,0.096627
1,Amazon ARIPL,718321,2026-06-30,-1,0,0,0.000000,0.000000,0.000000,0.000000
2,Amazon ARIPL,718322,2026-06-30,0,-1,0,0.000000,0.419733,0.141604,0.092710
3,Amazon ARIPL,718323,2026-06-30,-1,0,0,0.000000,0.000000,0.000000,0.000000
4,Amazon ARIPL,718328,2026-06-30,1,-1,0,0.000000,0.173154,0.055332,0.039274
...,...,...,...,...,...,...,...,...,...,...
3218,Nykaa,810673,2026-06-30,0,1,0,0.000808,0.004335,0.002219,0.000705
3219,Nykaa,810674,2026-06-30,0,1,0,0.000063,0.001645,0.000696,0.000316
3220,Nykaa,810738,2026-06-30,-1,0,0,0.000000,0.000607,0.000105,0.000168
3221,Nykaa,810805,2026-06-30,0,-1,0,0.000000,0.001446,0.000501,0.000315


In [258]:
# trend_df.to_csv('t_s_t_df_ecom.csv')

In [259]:
final_df = final_df.merge(seasonality_df, on = ["platform_name", "brand_code", 'run_month'], how = 'left')
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,qtr_ind_rate_x,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value,qtr_ind_rate_y,seasonality_flag
0,Amazon RK_718472,2023-11-30,0.0000,0.51,1.125,0.452544,0.765,0.000000,0.000025,0.000056,0.000022,0.000038,718472,Amazon RK,ADV-AHO-R,0.0,1.0,0.0,0.0,0.0,496.828458,0.000022,0.0,0.0,0.45,0.000022,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,0.676890,0.547432,0.45,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000034,0.000027,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.00000,0.0,0.0,0.0,0.0,496.828458,0
1,Meesho_718472,2025-12-31,0.0000,0.00,0.000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.002119,0.0,0.0,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,0.000000,0.000000,42.66,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1,0,0.0,0,1.26,9.45,0.000063,0.00047,NaN,NaN,NaN,NaN,496.828458,0
2,Amazon RK_718472,2023-12-31,0.4500,0.51,1.125,0.289509,0.774,0.000022,0.000025,0.000056,0.000014,0.000038,718472,Amazon RK,ADV-AHO-R,0.0,0.0,1.0,0.0,0.0,496.828458,0.000000,0.0,0.0,0.00,0.000000,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,0.545380,0.432579,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000027,0.000021,0.0,0.0,0.0,0.0,0.000022,0.000000,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.00000,0.0,0.0,0.0,0.0,496.828458,0
3,Meesho_718472,2026-01-31,0.0000,0.00,0.000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000456,0.0,0.0,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,0.000000,0.000000,9.18,42.66,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,42.66,0.002119,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.002119,0.000000,0.0,1,0,0.0,0,1.26,9.45,0.000063,0.00047,NaN,NaN,NaN,NaN,496.828458,0
4,Amazon RK_718472,2024-01-31,0.1725,0.51,1.125,1.143780,1.116,0.000009,0.000025,0.000056,0.000057,0.000055,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000054,0.0,0.0,1.08,0.000054,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,1.380028,1.239281,1.08,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000069,0.000062,0.0,0.0,0.0,0.0,0.000000,0.000022,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.00000,0.0,0.0,0.0,0.0,496.828458,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,

In [260]:
trend_df.columns

Index(['platform_name', 'parent_material_code', 'run_month', 'trend_flag',
       'p3m_slope_flag', 'final_trend', 'lower_threshold', 'upper_threshold',
       'mean_value', 'std_value'],
      dtype='object')

In [261]:
final_df = final_df.merge(trend_df[['platform_name', 'parent_material_code', 'run_month',
                                    'final_trend','lower_threshold', 'upper_threshold']], on = ["platform_name", "parent_material_code", 'run_month'], how = 'left')
final_df


,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,qtr_ind_rate_x,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value,qtr_ind_rate_y,seasonality_flag,final_trend,lower_threshold,upper_threshold
0,Amazon RK_718472,2023-11-30,0.0000,0.51,1.125,0.452544,0.765,0.000000,0.000025,0.000056,0.000022,0.000038,718472,Amazon RK,ADV-AHO-R,0.0,1.0,0.0,0.0,0.0,496.828458,0.000022,0.0,0.0,0.45,0.000022,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,0.676890,0.547432,0.45,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000034,0.000027,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.00000,0.0,0.0,0.0,0.0,496.828458,0,0,0.0,0.000000
1,Meesho_718472,2025-12-31,0.0000,0.00,0.000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.002119,0.0,0.0,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,0.000000,0.000000,42.66,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1,0,0.0,0,1.26,9.45,0.000063,0.00047,NaN,NaN,NaN,NaN,496.828458,0,-1,0.0,0.002728
2,Amazon RK_718472,2023-12-31,0.4500,0.51,1.125,0.289509,0.774,0.000022,0.000025,0.000056,0.000014,0.000038,718472,Amazon RK,ADV-AHO-R,0.0,0.0,1.0,0.0,0.0,496.828458,0.000000,0.0,0.0,0.00,0.000000,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,0.545380,0.432579,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000027,0.000021,0.0,0.0,0.0,0.0,0.000022,0.000000,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.00000,0.0,0.0,0.0,0.0,496.828458,0,0,0.0,0.000000
3,Meesho_718472,2026-01-31,0.0000,0.00,0.000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000456,0.0,0.0,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,0.000000,0.000000,9.18,42.66,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,42.66,0.002119,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.002119,0.000000,0.0,1,0,0.0,0,1.26,9.45,0.000063,0.00047,NaN,NaN,NaN,NaN,496.828458,0,-1,0.0,0.002728
4,Amazon RK_718472,2024-01-31,0.1725,0.51,1.125,1.143780,1.116,0.000009,0.000025,0.000056,0.000057,0.000055,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000054,0.0,0.0,1.08,0.000054,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,1.380028,1.239281,1.08,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000069,0.000062,0.0,0.0,0.0,0.0,0.000000,0.000022,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.00000,0.0,0.0,0.0,0.0,496.828458,0,0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...

In [262]:
final_df[final_df.select_dtypes(include='number').columns] = final_df.select_dtypes(include='number').fillna(0)
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,qtr_ind_rate_x,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value,qtr_ind_rate_y,seasonality_flag,final_trend,lower_threshold,upper_threshold
0,Amazon RK_718472,2023-11-30,0.0000,0.51,1.125,0.452544,0.765,0.000000,0.000025,0.000056,0.000022,0.000038,718472,Amazon RK,ADV-AHO-R,0.0,1.0,0.0,0.0,0.0,496.828458,0.000022,0.0,0.0,0.45,0.000022,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,0.676890,0.547432,0.45,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000034,0.000027,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.00000,0.0,0.0,0.0,0.0,496.828458,0,0,0.0,0.000000
1,Meesho_718472,2025-12-31,0.0000,0.00,0.000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.002119,0.0,0.0,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,0.000000,0.000000,42.66,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1,0,0.0,0,1.26,9.45,0.000063,0.00047,0.0,0.0,0.0,0.0,496.828458,0,-1,0.0,0.002728
2,Amazon RK_718472,2023-12-31,0.4500,0.51,1.125,0.289509,0.774,0.000022,0.000025,0.000056,0.000014,0.000038,718472,Amazon RK,ADV-AHO-R,0.0,0.0,1.0,0.0,0.0,496.828458,0.000000,0.0,0.0,0.00,0.000000,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,0.545380,0.432579,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000027,0.000021,0.0,0.0,0.0,0.0,0.000022,0.000000,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.00000,0.0,0.0,0.0,0.0,496.828458,0,0,0.0,0.000000
3,Meesho_718472,2026-01-31,0.0000,0.00,0.000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,718472,Meesho,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000456,0.0,0.0,0.00,0.000000,NaT,0.000000,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,0.000000,0.000000,9.18,42.66,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,42.66,0.002119,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.002119,0.000000,0.0,1,0,0.0,0,1.26,9.45,0.000063,0.00047,0.0,0.0,0.0,0.0,496.828458,0,-1,0.0,0.002728
4,Amazon RK_718472,2024-01-31,0.1725,0.51,1.125,1.143780,1.116,0.000009,0.000025,0.000056,0.000057,0.000055,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000054,0.0,0.0,1.08,0.000054,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-06-30,None,Hair Oils,1.380028,1.239281,1.08,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000069,0.000062,0.0,0.0,0.0,0.0,0.000000,0.000022,0.0,0,0,0.0,0,0.00,0.00,0.000000,0.00000,0.0,0.0,0.0,0.0,496.828458,0,0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...

In [263]:
final_df.drop(columns = ['big_billion_days', 'big_billion_days_lag_1', 'big_billion_days_lag_2',
       'big_billion_days_lead_1', 'big_billion_days_lead_2', 'great_indian_festival',
       'great_indian_festival_lag_1', 'great_indian_festival_lag_2',
       'great_indian_festival_lead_1', 'great_indian_festival_lead_2'], inplace = True)

In [264]:
final_df.columns

Index(['key', 'month_date', 'pred_SARIMA', 'pred_p3m', 'pred_p6m',
       'pred_prophet', 'pred_rf', 'pred_value_SARIMA', 'pred_value_p3m',
       'pred_value_p6m', 'pred_value_prophet', 'pred_value_rf',
       'parent_material_code', 'platform_name', 'brand_code', 'qtr_ind_rate_x',
       'vol_in_rum_value', 'pred_best_model', 'pred_value_best_model',
       'vol_in_rum_treated', 'vol_in_rum_value_treated', 'train_till', 'cov',
       'run', 'step', 'file_path', 'run_month', 'M month', 'portfolio',
       'pred_prophet_70%ile', 'pred_prophet_60%ile', 'vol_in_rum', 'P3M',
       'P6M', 'LY P3M', 'LY P6M', 'LY P3M_copy', 'P3M Max', 'P3M Top 2 Mean',
       'MoM P3M growth', 'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2',
       '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)', 'P3M_value',
       'P6M_value', 'LY P3M_value', 'LY P6M_value',
       'pred_prophet_70%ile_value', 'pred_prophet_60%ile_value', 'LY', 'LLY',
       'LY value', 'LLY value', 'OT_Value_in_Cr_lag_1', 'OT_Value_in_C

In [265]:
missing_df['LY P3M'].sum()

0.0

In [266]:
final_df = final_df.sort_values(['key', 'month_date'])

# base LY


# LY lags
final_df['ly_lag1_value'] = (
    final_df
    .groupby(['key'])['vol_in_rum_value']
    .shift(13)
)

final_df['ly_lag2_value'] = (
    final_df
    .groupby(['key'])['vol_in_rum_value']
    .shift(14)
)

# LY leads
final_df['ly_lead1_value'] = (
    final_df
    .groupby(['key'])['vol_in_rum_value']
    .shift(11)
)

final_df['ly_lead2_value'] = (
    final_df
    .groupby(['key'])['vol_in_rum_value']
    .shift(10)
)


In [267]:
final_df[final_df['month_date'] == '2026-04-30']['P3M_value'].sum()

41.457977550408735

In [268]:
brand_seas = pd.read_excel('/data/aman_singh/acuuracy_check/seasonality.xlsx', sheet_name = 'brand')
brand_seas.columns = brand_seas.columns.str.lower()
brand_seas.rename(columns={'brand':'brand_code', 'months_num':'month', 'flag':'is_seasonal_month'}, inplace=True)

final_df['month_date'] = pd.to_datetime(final_df['month_date'])
final_df['month'] = final_df['month_date'].dt.month
final_df = final_df.merge(brand_seas, on = ['brand_code', 'month'], how = 'left')
final_df['is_seasonal_month'].fillna(0, inplace=True)

psku_seas = pd.read_excel('/data/aman_singh/acuuracy_check/seasonality.xlsx', sheet_name = 'psku')
psku_seas.columns = psku_seas.columns.str.lower()
psku_seas.rename(columns={'months_num':'month', 'flag':'is_seasonal_month_psku'}, inplace=True)

final_df = final_df.merge(psku_seas[['parent_material_code', 'month','is_seasonal_month_psku']], on = ['parent_material_code', 'month'], how = 'left')
final_df['is_seasonal_month_psku'].fillna(0, inplace=True)
final_df['final_seasonal_month'] = np.where(
    (final_df['is_seasonal_month'] == 1) | (final_df['is_seasonal_month_psku'] == 1), 1, 0
)



In [269]:
final_df['run_month'] = pd.to_datetime(final_df['run_month'])
def compute_adjusted_pm(df_grp, window, column, year_shift=0):
    df_grp = df_grp.sort_values("month_date")

    run_month = df_grp["run_month"].iloc[0]

    # define cutoff
    end_date = run_month - pd.DateOffset(years=year_shift)

    # keep only eligible history (before run month & non-event)
    hist = df_grp[
        (df_grp["month_date"] < end_date) &
        (df_grp["final_seasonal_month"] == 0)
    ]

    if hist.empty:
        return np.nan

    # take last `window` non-event months
    hist = hist.tail(window)

    # if len(hist) < window:
    #     return np.nan   # optional, keeps behavior strict

    return hist[column].mean()

adj_df = final_df.groupby(
    ['key', "run_month"]
).apply(
    lambda x: pd.Series({
        "P3M_non_seasonal": compute_adjusted_pm(x, 3,'vol_in_rum',0),
        "P6M_non_seasonal": compute_adjusted_pm(x, 6,'vol_in_rum',0),
        "P3M_non_seasonal_value": compute_adjusted_pm(x, 3,'vol_in_rum_value',0),
        "P6M_non_seasonal_value": compute_adjusted_pm(x, 6,'vol_in_rum_value',0)
    })
).reset_index()
adj_df


adj_ly_df = final_df.groupby(
    ['key', "run_month"]
).apply(
    lambda x: pd.Series({
        "LY_P3M_non_seasonal": compute_adjusted_pm(x, 3,'vol_in_rum',year_shift=1),
        "LY_P6M_non_seasonal": compute_adjusted_pm(x, 6,'vol_in_rum', year_shift=1),
        "LY_P3M_non_seasonal_value": compute_adjusted_pm(x, 3,'vol_in_rum_value', year_shift=1),
        "LY_P6M_non_seasonal_value": compute_adjusted_pm(x, 6,"vol_in_rum_value", year_shift=1)
    })
).reset_index()

adj_df = adj_df.merge(adj_ly_df, on = ['key', 'run_month'], how = 'left')
#adj_df[adj_df['key'] == 'reliance_b2c_2_haryana_718488']
adj_df

,key,run_month,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value
0,Amazon ARIPL_718288,2026-06-30,27.226000,24.622000,0.378075,0.341914,13.094000,13.525000,0.181830,0.187815
1,Amazon ARIPL_718321,2026-06-30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,Amazon ARIPL_718322,2026-06-30,16.606667,12.237500,0.280366,0.206603,4.421667,4.796667,0.074650,0.080981
3,Amazon ARIPL_718323,2026-06-30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,Amazon ARIPL_718328,2026-06-30,9.683100,7.020450,0.119719,0.086799,1.764667,1.719167,0.021818,0.021255
...,...,...,...,...,...,...,...,...,...,...
3218,Nykaa_810673,2026-06-30,1.801333,1.820000,0.002317,0.002341,NaN,NaN,NaN,NaN
3219,Nykaa_810674,2026-06-30,0.522667,0.464333,0.000672,0.000597,NaN,NaN,NaN,NaN
3220,Nykaa_810738,2026-06-30,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN
3221,Nykaa_810805,2026-06-30,14.842000,13.683600,0.000544,0.000501,NaN,NaN,NaN,NaN


In [270]:
final_df.shape

(122793, 83)

In [271]:
#adj_df.to_csv('seasonal_p3m_qcom.csv', index=False)
#all[all['month_date'].isin(['2025-11-30','2025-12-31','2026-01-31'])].groupby(['key','run_month','month_date','final_seasonal_month'])['vol_in_rum'].sum().reset_index().to_csv('seasonal_month_check.csv', index=False)
final_df = final_df.merge(
    adj_df,
    on=['key'],
    how="left"
)
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,qtr_ind_rate_x,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month_x,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value,qtr_ind_rate_y,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value,month,is_seasonal_month,seasonal_months,is_seasonal_month_psku,final_seasonal_month,run_month_y,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value
0,Amazon ARIPL_718288,2023-01-31,0.397728,9.020000,9.154167,9.113322,9.397314,0.005523,0.125256,0.12712,0.126552,0.130496,718288,Amazon ARIPL,SAFF GOLD,138865.260689,0.133866,0.0,0.0,9.640,0.133866,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,10.770226,9.992578,9.640,0.000000,0.000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.149561,0.138762,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0,0,0.0,1,27.226,24.622,0.378075,0.341914,13.094,13.525,0.18183,0.187815,138865.260689,0,0,0.104661,0.587795,NaN,NaN,NaN,NaN,1,0.0,NaN,0.0,0,2026-06-30,27.226,24.622,0.378075,0.341914,13.094,13.525,0.18183,0.187815
1,Amazon ARIPL_718288,2023-02-28,10.037441,9.020000,9.154167,7.401517,8.737608,0.139385,0.125256,0.12712,0.102781,0.121335,718288,Amazon ARIPL,SAFF GOLD,138865.260689,0.109912,0.0,0.0,7.915,0.109912,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,8.748662,7.926816,7.915,0.000000,0.000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.121489,0.110076,0.0,0.0,0.0,0.0,0.133866,0.000000,0.000000,0,0,0.0,1,27.226,24.622,0.378075,0.341914,13.094,13.525,0.18183,0.187815,138865.260689,0,0,0.104661,0.587795,NaN,NaN,NaN,NaN,2,0.0,NaN,0.0,0,2026-06-30,27.226,24.622,0.378075,0.341914,13.094,13.525,0.18183,0.187815
2,Amazon ARIPL_718288,2023-03-31,9.856017,9.020000,9.154167,12.787684,9.508410,0.136866,0.125256,0.12712,0.177577,0.132039,718288,Amazon ARIPL,SAFF GOLD,138865.260689,0.131991,0.0,0.0,9.505,0.131991,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,14.499796,13.634569,9.505,0.000000,0.000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.201352,0.189337,0.0,0.0,0.0,0.0,0.109912,0.133866,0.000000,0,0,0.0,1,27.226,24.622,0.378075,0.341914,13.094,13.525,0.18183,0.187815,138865.260689,0,0,0.104661,0.587795,NaN,NaN,NaN,NaN,3,0.0,NaN,0.0,0,2026-06-30,27.226,24.622,0.378075,0.341914,13.094,13.525,0.18183,0.187815
3,Amazon ARIPL_718288,2023-04-30,9.579111,9.020000,9.154167,4.585437,9.076793,0.133021,0.125256,0.12712,0.063676,0.126045,718288,Amazon ARIPL,SAFF GOLD,138865.260689,0.129006,0.0,0.0,9.290,0.129006,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,6.055104,5.292286,9.290,9.020000,0.000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,9.020000,0.125256,0.000000,0.0,0.0,0.084084,0.073491,0.0,0.0,0.0,0.0,0.131991,0.109912,0.133866,0,0,0.0,1,27.226,24.622,0.378075,0.341914,13

In [272]:
mnth = 7
all_brand = final_df.groupby(['brand_code', 'run_month_x','month_date'])['vol_in_rum_value'].sum().reset_index()
def detect_april_anomaly(df, brand_code, threshold=0.25, months_window=3):
    """
    Detect if April's vol_in_rum_value is >25% different 
    from past 3 months & next 3 months, and if pattern repeats in last 2 years.
    
    Parameters:
    - df: input dataframe with 'month_date', 'vol_in_rum_value', 'run_month'
    - brand_code: filter by this brand code
    - threshold: 25% difference threshold
    - months_window: number of months before and after April to compare
    
    """
    
    # Filter for brand and sort by month_date
    df_brand = df[df['brand_code'] == brand_code].sort_values('month_date').copy()
    
    if df_brand.empty:
        return None
    
    # Extract year and month
    df_brand['year'] = df_brand['month_date'].dt.year
    df_brand['month'] = df_brand['month_date'].dt.month
    
    # Get unique years (excluding current year if incomplete)
    years = sorted(df_brand['year'].unique())
    current_year = years[-1]
    past_years = [y for y in years if y < current_year][-2:]  # Last 2 years
    
    anomalies = []
    
    # Check each past year's April
    for year in past_years:
        df_year = df_brand[df_brand['year'] == year].sort_values('month_date')
        
        # Get April data (month == 4)
        april_data = df_year[df_year['month'] == mnth]
        if april_data.empty:
            continue
        
        april_value = april_data['vol_in_rum_value'].iloc[0]
        april_month = mnth
        
        # Dynamically calculate past and next months
        past_months = [(april_month - i - 1) % 12 + 1 for i in range(1,months_window+1)]
        print(past_months)
        next_months = [(april_month + i - 1) % 12 + 1 for i in range(1, months_window + 1)]
        print(next_months)
        
        # Get past and next months values
        past_3m = df_year[df_year['month'].isin(past_months)]['vol_in_rum_value']
        next_3m = df_year[df_year['month'].isin(next_months)]['vol_in_rum_value']
        
        # Combine all comparison months
        comparison_values = pd.concat([past_3m, next_3m])
        
        if comparison_values.empty:
            continue
        
        # Calculate mean of comparison months
        #mean_value = comparison_values.mean()
        
        # Calculate percentage difference
        pct_diffs = []
        for comp_value in comparison_values:
            if comp_value != 0:
                pct_diff = (april_value - comp_value) / comp_value
                pct_diffs.append(pct_diff)
        
        # April is anomalous if it's >25% different from ALL comparison months
        # AND all differences have the same sign (all positive or all negative)
        if pct_diffs:
            positive_diffs = [p for p in pct_diffs if p > 0]
            negative_diffs = [p for p in pct_diffs if p < 0]
            same_sign = len(positive_diffs) == len(pct_diffs) or len(negative_diffs) == len(pct_diffs)
            is_anomaly = len([p for p in pct_diffs if abs(p) > threshold]) == len(pct_diffs) and same_sign
        else:
            is_anomaly = False

        anomalies.append({
            'brand_code': brand_code,
            'year': year,
            'april_value': april_value,
            'num_months_compared': len(comparison_values),
            'pct_diffs_from_each': pct_diffs,
            'min_pct_diff': min(pct_diffs) * 100 if pct_diffs else None,
            'max_pct_diff': max(pct_diffs) * 100 if pct_diffs else None,
            'is_anomaly': is_anomaly,
            'direction': 'higher' if april_value > comparison_values.mean() else 'lower'
        })
    
    # Check if pattern repeats in both years
    if len(anomalies) == 2:
        pattern_repeats = anomalies[0]['is_anomaly'] and anomalies[1]['is_anomaly']
        return pd.DataFrame(anomalies), pattern_repeats
    
    return pd.DataFrame(anomalies), False


# Usage: Apply to each brand code
brands = all_brand['brand_code'].unique()
results = []

for brand in brands:
    df_result, repeats = detect_april_anomaly(all_brand, brand)
    if df_result is not None and not df_result.empty:
        df_result['pattern_repeats'] = repeats
        results.append(df_result)


anomaly_summary = pd.concat(results, ignore_index=True)
print(anomaly_summary[anomaly_summary['pattern_repeats'] == True])

[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8, 9, 10]
[6, 5, 4]
[8,

In [273]:
final_brands = anomaly_summary[anomaly_summary['pattern_repeats'] == True].drop_duplicates(subset=['brand_code'])[['brand_code', 'direction', 'min_pct_diff', 'max_pct_diff']]
final_brands['month_different'] = 1
final_df = final_df.merge(final_brands[['brand_code', 'month_different']], on = 'brand_code', how = 'left')
final_df['month_different'].fillna(0, inplace=True)
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,qtr_ind_rate_x,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month_x,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value,qtr_ind_rate_y,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value,month,is_seasonal_month,seasonal_months,is_seasonal_month_psku,final_seasonal_month,run_month_y,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value,month_different
0,Amazon ARIPL_718288,2023-01-31,0.397728,9.020000,9.154167,9.113322,9.397314,0.005523,0.125256,0.12712,0.126552,0.130496,718288,Amazon ARIPL,SAFF GOLD,138865.260689,0.133866,0.0,0.0,9.640,0.133866,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,10.770226,9.992578,9.640,0.000000,0.000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.149561,0.138762,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0,0,0.0,1,27.226,24.622,0.378075,0.341914,13.094,13.525,0.18183,0.187815,138865.260689,0,0,0.104661,0.587795,NaN,NaN,NaN,NaN,1,0.0,NaN,0.0,0,2026-06-30,27.226,24.622,0.378075,0.341914,13.094,13.525,0.18183,0.187815,0.0
1,Amazon ARIPL_718288,2023-02-28,10.037441,9.020000,9.154167,7.401517,8.737608,0.139385,0.125256,0.12712,0.102781,0.121335,718288,Amazon ARIPL,SAFF GOLD,138865.260689,0.109912,0.0,0.0,7.915,0.109912,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,8.748662,7.926816,7.915,0.000000,0.000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.121489,0.110076,0.0,0.0,0.0,0.0,0.133866,0.000000,0.000000,0,0,0.0,1,27.226,24.622,0.378075,0.341914,13.094,13.525,0.18183,0.187815,138865.260689,0,0,0.104661,0.587795,NaN,NaN,NaN,NaN,2,0.0,NaN,0.0,0,2026-06-30,27.226,24.622,0.378075,0.341914,13.094,13.525,0.18183,0.187815,0.0
2,Amazon ARIPL_718288,2023-03-31,9.856017,9.020000,9.154167,12.787684,9.508410,0.136866,0.125256,0.12712,0.177577,0.132039,718288,Amazon ARIPL,SAFF GOLD,138865.260689,0.131991,0.0,0.0,9.505,0.131991,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,14.499796,13.634569,9.505,0.000000,0.000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.201352,0.189337,0.0,0.0,0.0,0.0,0.109912,0.133866,0.000000,0,0,0.0,1,27.226,24.622,0.378075,0.341914,13.094,13.525,0.18183,0.187815,138865.260689,0,0,0.104661,0.587795,NaN,NaN,NaN,NaN,3,0.0,NaN,0.0,0,2026-06-30,27.226,24.622,0.378075,0.341914,13.094,13.525,0.18183,0.187815,0.0
3,Amazon ARIPL_718288,2023-04-30,9.579111,9.020000,9.154167,4.585437,9.076793,0.133021,0.125256,0.12712,0.063676,0.126045,718288,Amazon ARIPL,SAFF GOLD,138865.260689,0.129006,0.0,0.0,9.290,0.129006,2026-05-31,0.473274,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,6.055104,5.292286,9.290,9.020000,0.000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,9.020000,0.125256,0.000000,0.0,0.0,0.084084,0.073491,0.0,0.0,0.0,0.0,0.131991,0.109912,0.133866,0,0,0.0,1,27.226

In [287]:
final_df[(final_df['M month'].notna())].to_csv('/data/aman_singh/acuuracy_check/all_combination_ecom_jun_pred.csv')

In [288]:
final_df.to_csv('/data/aman_singh/acuuracy_check/all_combination_ecom_trend.csv')

In [286]:
final_df[final_df['month_date'] == '2026-07-31']['pred_value_SARIMA'].sum()

32.81407409514513

### some checks

In [274]:
query = f"""select * from {input_table}
where run_month = '2026-06-30' """

data = pd.read_sql(con=dev_conn, sql=query)
data.columns = data.columns.str.lower()
data

,month_date,platform_name,parent_material_code,brand_code,vol_in_rum,indexbpm,imputed,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,run_month
0,2023-01-31,Amazon ARIPL,718288,SAFF GOLD,9.640,13.27071,0,0,0,0,0,0,0,0,0,0,0,2026-06-30
1,2023-02-28,Amazon ARIPL,718288,SAFF GOLD,7.915,10.89602,0,0,0,0,0,0,0,0,0,0,0,2026-06-30
2,2023-03-31,Amazon ARIPL,718288,SAFF GOLD,9.505,13.08486,0,0,0,0,0,0,0,0,0,0,0,2026-06-30
3,2023-04-30,Amazon ARIPL,718288,SAFF GOLD,9.290,12.78889,0,0,0,0,0,0,0,0,0,0,0,2026-06-30
4,2023-05-31,Amazon ARIPL,718288,SAFF GOLD,8.050,11.08187,0,0,0,0,0,0,0,0,0,0,0,2026-06-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127998,2026-11-30,Nykaa,811019,PADV_WIPS,0.000,0.00000,1,0,0,0,0,0,0,0,0,0,0,2026-06-30
127999,2026-12-31,Nykaa,811019,PADV_WIPS,0.000,0.00000,1,0,0,0,0,0,0,0,0,0,0,2026-06-30
128000,2027-01-31,Nykaa,811019,PADV_WIPS,0.000,0.00000,1,0,0,0,0,0,0,0,0,0,0,2026-06-30
128001,2027-02-28,Nykaa,811019,PADV_WIPS,0.000,0.00000,1,0,0,0,0,0,0,0,0,0,0,2026-06-30


In [275]:
data['key'] = (
    data['platform_name'].astype(str) + '_' +
    data['parent_material_code'].astype(str)
)
data = data[data['key'].isin(trend_file_df['key'].unique())]

In [276]:
trend_file_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3
0,Amazon RK_718472,2023-11-30,0.000000,0.51,1.125,0.452544,0.765,0.000000,0.000025,0.000056,0.000022,0.000038,718472,Amazon RK,ADV-AHO-R,0.0,1.0,0.0,0.0,0.0,496.828458,0.000022,NaN,NaN,0.45,0.000022,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Hair Oils,0.676890,0.547432,0.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000034,0.000027,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Amazon RK_718472,2023-12-31,0.450000,0.51,1.125,0.289509,0.774,0.000022,0.000025,0.000056,0.000014,0.000038,718472,Amazon RK,ADV-AHO-R,0.0,0.0,1.0,0.0,0.0,496.828458,0.000000,NaN,NaN,0.00,0.000000,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Hair Oils,0.545380,0.432579,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000027,0.000021,NaN,NaN,NaN,NaN,0.000022,NaN,NaN
2,Amazon RK_718472,2024-01-31,0.172500,0.51,1.125,1.143780,1.116,0.000009,0.000025,0.000056,0.000057,0.000055,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000054,NaN,NaN,1.08,0.000054,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Hair Oils,1.380028,1.239281,1.08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000069,0.000062,NaN,NaN,NaN,NaN,0.000000,0.000022,NaN
3,Amazon RK_718472,2024-02-29,0.678074,0.51,1.125,1.165967,1.296,0.000034,0.000025,0.000056,0.000058,0.000064,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000080,NaN,NaN,1.62,0.000080,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Hair Oils,1.416077,1.293417,1.62,0.51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.51,0.000025,NaN,NaN,NaN,0.000070,0.000064,NaN,NaN,NaN,NaN,0.000054,0.000000,0.000022
4,Amazon RK_718472,2024-03-31,0.844506,0.90,1.125,1.783082,1.818,0.000042,0.000045,0.000056,0.000089,0.000090,718472,Amazon RK,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,496.828458,0.000134,NaN,NaN,2.70,0.000134,2026-05-31,2.566769,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-06-30,None,Hair Oils,2.008802,1.887959,2.70,0.90,NaN,NaN,NaN,NaN,NaN,NaN,76.470588,NaN,NaN,NaN,0.90,0.000045,NaN,NaN,NaN,0.000100,0.000094,NaN,NaN,NaN,NaN,0.000080,0.000054,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111634,Big Basket_719192,2026-09-30,0.000000,0.00,0.000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,719192,Big Basket,VEG_CLEAN,NaN,NaN,NaN,NaN,NaN,100.000000,0.000000,NaN,NaN,0.00,0.000000,2026-05-31,4.605489,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,

In [277]:
import pandas as pd

as_of_date = pd.to_datetime("2026-06-30")  # month-end for Feb 2026
data['month_date'] = pd.to_datetime(data['month_date'])
filtered = data[
    
    (data['month_date'] < as_of_date) &
    (data['month_date'] >= as_of_date - pd.DateOffset(months=3))
]


In [278]:
# assert p3m equals
x = filtered.groupby(['month_date'])['vol_in_rum'].sum().reset_index()['vol_in_rum'].mean()
y = trend_file_df[trend_file_df['month_date'] == '2026-06-30']['P3M'].sum()
assert(int(x)==int(y))

In [279]:
(x,y)

(267025.7059451062, 267025.7059451031)

In [280]:
ly_end = as_of_date - pd.DateOffset(years=1)
ly_start = ly_end - pd.DateOffset(months=3)

filtered = data[
    
    (data['month_date'] < ly_end) &
    (data['month_date'] >= ly_start )
]


In [282]:
# p3m ly check may not equal but should be close
x = filtered.groupby(['month_date'])['vol_in_rum'].sum().reset_index()['vol_in_rum'].mean()
y = trend_file_df[trend_file_df['month_date'] == '2026-06-30']['LY P3M'].sum()
(x,y)

(307007.49542052665, 305203.2554095226)

In [283]:
# p3m consistency check
as_of_date = pd.to_datetime('2026-06-30')

next_3_months = pd.date_range(
    start=as_of_date + pd.offsets.MonthEnd(1),
    periods=3,
    freq='M'
)
for dt in next_3_months:
    p3m_sum = trend_file_df.loc[
        trend_file_df['month_date'] == dt, 'P3M'
    ].sum()
    
    print(f"P3M sum for {dt.date()}: {p3m_sum}")


P3M sum for 2026-07-31: 267025.7059451031
P3M sum for 2026-08-31: 267025.7059451031
P3M sum for 2026-09-30: 267025.7059451031


### The end

In [106]:
import numpy as np
import pandas as pd

df = final_df.copy()

# --------------------------------------------------
# 1. SAFE FACTOR FUNCTIONS (NO ERRORS)
# --------------------------------------------------

def safe_div(a, b):
    """Safe division: if error or b<=0 → return 1."""
    try:
        if b is None or b == 0:
            return 1
        return a / b
    except:
        return 1

# recency factor = p3m/p6m (cap at 2)
df["recency_factor"] = df.apply(
    lambda r: min(2, safe_div(r["P3M_value"], r["P6M_value"])),
    axis=1
)

def safe_shrink(r,column_name):
    try:
        ratio = r[column_name] / r["P3M_value"]
        return 1 / np.sqrt(ratio)
    except:
        return 1

df["shrink_ratio_prophet"] = df.apply(lambda r: safe_shrink(r, "pred_value_prophet"), axis=1)
df["shrink_ratio_rf"] = df.apply(lambda r: safe_shrink(r, "pred_value_rf"), axis=1)

# seasonality factor = p3m / p3mLY (cap at 2)
df["seasonality_factor"] = df.apply(
    lambda r: min(2, safe_div(r["P3M_value"], r["LY P3M_value"])),
    axis=1
)


# shrink_ratio = 1 / sqrt(max(forecast/p3m,1)) → safe



# --------------------------------------------------
# 2. HEURISTICS
# --------------------------------------------------

# Recency heuristic: max(recency_factor * P3M, forecast)
df["recency_heuristic_prophet_value"] = df.apply(
    lambda r: max(r["recency_factor"] * r["P3M_value"], r["pred_value_prophet"]),
    axis=1
)

# Seasonality heuristic: max(seasonality_value, forecast)
df["seasonality_heuristic_prophet_value"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY value'], r["pred_value_prophet"]),
    axis=1
)

# Recency + Seasonality combined
df["recency_seasonality_heuristic_prophet_value"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY value'] * r["recency_factor"],
                  r["pred_value_prophet"]),
    axis=1
)

# Non-seasonal heuristic
df["non_seasonal_heuristic_prophet_value"] = df["pred_value_prophet"] * df["shrink_ratio_prophet"]


# --------------------------------------------------
# 3. FINAL DECISION TREE + skipped condition
# --------------------------------------------------

def apply_final_logic(r):

    # Rule 1: skipped → force P3M
    if r.get("skipped", 0) == 1:
        return r["P3M_value"]

    
    # Rule 3: Heuristic combinations
    if r["final_trend"] == 1 and r["seasonality_flag"] == 0:
        return r["recency_heuristic_prophet_value"]


    if r["final_trend"] != 1 and r["seasonality_flag"] == 1:
        return r["seasonality_heuristic_prophet_value"]
    
    if r["final_trend"] == 1 and r["seasonality_flag"] == 1:
        return r["recency_seasonality_heuristic_prophet_value"]

    # Default: (0,0)
    return r["non_seasonal_heuristic_prophet_value"]


df["final_heuristic_prophet_value"] = df.apply(apply_final_logic, axis=1)


In [107]:
df["recency_heuristic_rf_value"] = df.apply(
    lambda r: max(r["recency_factor"] * r["P3M_value"], r["pred_value_rf"]),
    axis=1
)

# Seasonality heuristic: max(seasonality_value, forecast)
df["seasonality_heuristic_rf_value"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY value'], r["pred_value_rf"]),
    axis=1
)

# Recency + Seasonality combined
df["recency_seasonality_heuristic_rf_value"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY value'] * r["recency_factor"],
                  r["pred_value_rf"]),
    axis=1
)

# Non-seasonal heuristic
df["non_seasonal_heuristic_rf_value"] = df["pred_value_rf"] * df["shrink_ratio_rf"]


# --------------------------------------------------
# 3. FINAL DECISION TREE + skipped condition
# --------------------------------------------------

def apply_final_logic(r):

    # Rule 1: skipped → force P3M
    if r.get("skipped", 0) == 1:
        return r["P3M_value"]

    
    # Rule 3: Heuristic combinations
    if r["final_trend"] == 1 and r["seasonality_flag"] == 0:
        return r["recency_heuristic_rf_value"]


    if r["final_trend"] != 1 and r["seasonality_flag"] == 1:
        return r["seasonality_heuristic_rf_value"]
    
    if r["final_trend"] == 1 and r["seasonality_flag"] == 1:
        return r["recency_seasonality_heuristic_rf_value"]

    # Default: (0,0)
    return r["non_seasonal_heuristic_rf_value"]


df["final_heuristic_rf_value"] = df.apply(apply_final_logic, axis=1)


In [108]:
df["error_prophet"] = df["pred_value_prophet"] - df["vol_in_rum_value"]
df["abs_error_prophet"] = df["error_prophet"].abs()

df["error_rf"] = df["pred_value_rf"] - df["vol_in_rum_value"]
df["abs_error_rf"] = df["error_rf"].abs()

df["error_final_heuristic_prophet"] = df["final_heuristic_prophet_value"] - df["vol_in_rum_value"]
df["abs_error_final_heuristic_prophet"] = df["error_final_heuristic_prophet"].abs()

df["error_final_heuristic_rf"] = df["final_heuristic_rf_value"] - df["vol_in_rum_value"]
df["abs_error_final_heuristic_rf"] = df["error_final_heuristic_rf"].abs()


In [109]:
df["recency_heuristic_rf"] = df.apply(
    lambda r: max(r["recency_factor"] * r["P3M"], r["pred_rf"]),
    axis=1
)

# Seasonality heuristic: max(seasonality_value, forecast)
df["seasonality_heuristic_rf"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY'], r["pred_rf"]),
    axis=1
)

# Recency + Seasonality combined
df["recency_seasonality_heuristic_rf"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY'] * r["recency_factor"],
                  r["pred_rf"]),
    axis=1
)

# Non-seasonal heuristic
df["non_seasonal_heuristic_rf"] = df["pred_rf"] * df["shrink_ratio_rf"]


# --------------------------------------------------
# 3. FINAL DECISION TREE + skipped condition
# --------------------------------------------------

def apply_final_logic(r):

    # Rule 1: skipped → force P3M
    if r.get("skipped", 0) == 1:
        return r["P3M_value"]

    
    # Rule 3: Heuristic combinations
    if r["final_trend"] == 1 and r["seasonality_flag"] == 0:
        return r["recency_heuristic_rf"]


    if r["final_trend"] != 1 and r["seasonality_flag"] == 1:
        return r["seasonality_heuristic_rf"]
    
    if r["final_trend"] == 1 and r["seasonality_flag"] == 1:
        return r["recency_seasonality_heuristic_rf"]

    # Default: (0,0)
    return r["non_seasonal_heuristic_rf"]


df["final_heuristic_rf"] = df.apply(apply_final_logic, axis=1)


In [110]:
df["recency_heuristic_prophet"] = df.apply(
    lambda r: max(r["recency_factor"] * r["P3M"], r["pred_prophet"]),
    axis=1
)

# Seasonality heuristic: max(seasonality_value, forecast)
df["seasonality_heuristic_prophet"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY'], r["pred_prophet"]),
    axis=1
)

# Recency + Seasonality combined
df["recency_seasonality_heuristic_prophet"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY'] * r["recency_factor"],
                  r["pred_prophet"]),
    axis=1
)

# Non-seasonal heuristic
df["non_seasonal_heuristic_prophet"] = df["pred_prophet"] * df["shrink_ratio_prophet"]


# --------------------------------------------------
# 3. FINAL DECISION TREE + skipped condition
# --------------------------------------------------

def apply_final_logic(r):

    # Rule 1: skipped → force P3M
    if r.get("skipped", 0) == 1:
        return r["P3M_value"]

    
    # Rule 3: Heuristic combinations
    if r["final_trend"] == 1 and r["seasonality_flag"] == 0:
        return r["recency_heuristic_prophet"]


    if r["final_trend"] != 1 and r["seasonality_flag"] == 1:
        return r["seasonality_heuristic_prophet"]
    
    if r["final_trend"] == 1 and r["seasonality_flag"] == 1:
        return r["recency_seasonality_heuristic_prophet"]

    # Default: (0,0)
    return r["non_seasonal_heuristic_prophet"]


df["final_heuristic_prophet"] = df.apply(apply_final_logic, axis=1)


In [112]:
df.to_csv('Heuristics_all_combination_ecom_cp.csv')

In [114]:
df[(df['M month'].notna())].to_csv('Heuristics_all_combination_ecom_cp2.csv')

In [2]:
import pandas as pd
final_df = pd.read_csv('Heuristics_all_combination_ecom_cp.csv')

/tmp/ipykernel_2052077/3783379672.py:2: DtypeWarning: Columns (20,22,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  final_df = pd.read_csv('Heuristics_all_combination_ecom_cp.csv')


In [6]:
import numpy as np
import pandas as pd

def compute_thresholds(df_grp):
    """
    df_grp MUST contain:
    - month_date
    - vol_in_rum
    - run_month

    Returns: lower_threshold, upper_threshold, mean, std
    """

    df_grp = df_grp.sort_values("month_date")
    run_month = df_grp["run_month"].max()

    # --- Use ONLY actual data (strictly before run month)
    df_actual = df_grp[df_grp["month_date"] < run_month]

    series = df_actual["vol_in_rum_value"].astype(float).values

    # If no real data → return zeros
    if len(series) == 0:
        return pd.Series({
            "lower_threshold": 0,
            "upper_threshold": 0,
            "mean_value": 0,
            "std_value": 0
        })

    # --- Take last 12 months OR all available
    if len(series) > 12:
        series = series[-12:]

    mean_val = np.mean(series)
    std_val = np.std(series)

    # --- SPECIAL CASE: ≤3 data points
    if len(series) <= 3:
        lower = 0.5 * mean_val
        upper = 2 * mean_val

        return pd.Series({
            "lower_threshold": lower,
            "upper_threshold": upper,
            "mean_value": mean_val,
            "std_value": std_val
        })

    # --- Normal case (std can be zero also)
    lower = max(0,mean_val - 2*std_val)
    upper = mean_val + 3*std_val

    return pd.Series({
        "lower_threshold": lower,
        "upper_threshold": upper,
        "mean_value": mean_val,
        "std_value": std_val
    })

threshold_df = final_df.groupby(
    ["platform_name", "parent_material_code", "run_month"]
).apply(compute_thresholds).reset_index()

threshold_df.head()


,platform_name,parent_material_code,run_month,lower_threshold,upper_threshold,mean_value,std_value
0,Amazon ARIPL,718288,2025-12-31,0.134916,0.345766,0.219256,0.042170
1,Amazon ARIPL,718321,2025-12-31,0.000000,0.000000,0.000000,0.000000
2,Amazon ARIPL,718322,2025-12-31,0.060035,0.108314,0.079347,0.009656
3,Amazon ARIPL,718323,2025-12-31,0.000000,0.000000,0.000000,0.000000
4,Amazon ARIPL,718328,2025-12-31,0.016588,0.032115,0.022799,0.003105


In [7]:
threshold_df.to_csv('ecom_threshold.csv')

In [162]:
final_df[(final_df['M month'].notna())].to_csv('Heuristics_all_combination_qcom_cp_chk.csv')

In [129]:
final_df[final_df['month_date'] == '2026-02-28']['LY P3M_value'].sum()

123.92211585241307

In [49]:
df['TREND'].unique()

array([ 1,  0, -1])

In [96]:
prophet_output = collate_file('prophet_data_train_till')

downloaded_results\new_pipeline\202508-09_FK_Others_Offtakes\train_till_30_Jun_2025\prophet_results\prophet_data_train_till_30_Jun_2025.csv
downloaded_results\new_pipeline\202508-09_FK_Others_Offtakes\train_till_31_Jul_2025\prophet_results\prophet_data_train_till_31_Jul_2025.csv
downloaded_results\new_pipeline\202509_AZ_BB_Offtakes\train_till_30_Jun_2025\prophet_results\prophet_data_train_till_30_Jun_2025.csv
downloaded_results\new_pipeline\202509_AZ_BB_Offtakes\train_till_31_Jul_2025\prophet_results\prophet_data_train_till_31_Jul_2025.csv


In [100]:
len_before_merge = len(offtake_df)
offtake_df = offtake_df.merge(
    read_qtr_ind_rate_table()[['brand_code', 'qtr_ind_rate']] ,
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(offtake_df)


Credentials retrieved successfully for prod db.


In [ ]:
offtake_df

In [104]:
offtake_df['OT_Value_in_Cr'] = offtake_df['vol_in_rum'] * offtake_df['qtr_ind_rate'] / (10 ** 7)

In [105]:
offtake_df.to_csv('Offtake_realigned_base.csv', index=False)

In [108]:
trend_file_df[trend_file_df['run_month'].isin(['2025-10-31'])].to_csv('Trend_File_Oct_Live_Run_OT.csv', index=False)

In [107]:
trend_file_df[trend_file_df['run_month'].isin(['2025-07-31', '2025-08-31'])].to_csv('Trend_File_SepAug_OT.csv', index=False)

In [97]:
prophet_output.to_csv('ECOM_Prophet_Trend_OT_Chain_PSKU_AugSep.csv', index=False)

In [81]:
forecast_df = trend_file_df[trend_file_df['month_date'] >= trend_file_df['run_month']]

forecast_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,LY P6M,P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,LY,LLY,LY value,LLY value
26,Amazon_718589,2025-03-31,1312.900000,1293.400000,1824.634404,945.661643,0.059090,0.058213,0.082122,0.042562,...,841.200000,0.059090,0.058213,0.027878,0.037860,0.082122,717.3,1428.0,0.032284,0.064271
27,Amazon_718589,2025-04-30,1312.900000,1293.400000,1894.995393,1306.938750,0.059090,0.058213,0.085289,0.058822,...,792.800000,0.059090,0.058213,0.036843,0.035682,0.085289,1053.0,1040.4,0.047393,0.046826
28,Amazon_718589,2025-05-31,1312.900000,1293.400000,2055.487818,1194.591429,0.059090,0.058213,0.092512,0.053765,...,682.250000,0.059090,0.058213,0.039944,0.030706,0.092512,1241.1,1236.3,0.055859,0.055643
29,Amazon_718589,2025-06-30,1312.900000,1293.400000,1713.411024,989.591786,0.059090,0.058213,0.077116,0.044539,...,811.600000,0.059090,0.058213,0.045178,0.036528,0.077116,797.4,1176.0,0.035889,0.052929
56,Amazon_722188,2025-03-31,0.666667,6.400000,0.000000,471.682496,0.000030,0.000288,0.000000,0.021229,...,782.533333,0.000030,0.000288,0.043045,0.035220,0.000000,806.4,236.8,0.036294,0.010658
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120425,Flipkart National_809422,2025-10-31,211.833333,159.550000,262.584032,129.597491,0.030576,0.023029,0.037901,0.018706,...,207.383333,0.030576,0.023029,0.009040,0.029934,0.046132,0.8,NaN,0.000115,NaN
120442,Flipkart National_809423,2025-07-31,75.000000,53.533333,0.000000,169.044957,0.010826,0.007727,0.000000,0.024400,...,NaN,0.010826,0.007727,0.057582,NaN,0.008194,243.4,NaN,0.035132,NaN
120443,Flipkart National_809423,2025-08-31,75.000000,53.533333,0.000000,117.175693,0.010826,0.007727,0.000000,0.016913,...,NaN,0.010826,0.007727,0.054599,NaN,0.000000,99.6,NaN,0.014376,NaN
120444,Flipkart National_809423,2025-09-30,75.000000,53.533333,0.000000,111.920263,0.010826,0.007727,0.000000,0.016155,...,286.050000,0.010826,0.007727,0.048200,0.041288,0.000000,75.4,NaN,0.010883,NaN


In [98]:
trend_file_df[trend_file_df['run_month'] == '2025-08-31'].to_csv('Trend_file_Sep.csv', index=False)

In [82]:
forecast_df['is_na'] = forecast_df['vol_in_rum'].isna()
forecast_df.groupby(['run_month', 'month_date'])['is_na'].sum()

run_month   month_date
2025-03-31  2025-03-31    0
            2025-04-30    0
            2025-05-31    0
            2025-06-30    0
2025-04-30  2025-04-30    0
            2025-05-31    0
            2025-06-30    0
            2025-07-31    0
2025-05-31  2025-05-31    0
            2025-06-30    0
            2025-07-31    0
            2025-08-31    0
2025-06-30  2025-06-30    0
            2025-07-31    0
            2025-08-31    0
            2025-09-30    0
2025-07-31  2025-07-31    0
            2025-08-31    0
            2025-09-30    0
            2025-10-31    0
Name: is_na, dtype: int64

In [83]:
forecast_df.drop('is_na', axis=1, inplace=True)

In [84]:
pred_value_cols = [col for col in trend_file_df.columns if 'value' in col and 'pred' in col]
pred_value_cols

['pred_value_p3m',
 'pred_value_p6m',
 'pred_value_prophet',
 'pred_value_rf',
 'pred_value_best_model',
 'pred_prophet_70%ile_value']

In [85]:
for col in pred_value_cols:
    trend_file_df[f'error_{col}'] = trend_file_df[col] - trend_file_df['vol_in_rum_value']
    trend_file_df[f'abs_error_{col}'] = np.abs(trend_file_df[col] - trend_file_df['vol_in_rum_value'])    

In [86]:
trend_file_df.to_csv('Trend_file_OT_FK_AZ_BB.csv', index=False)

In [85]:
feature_importance_df = collate_file('feature_importance_train_till')

downloaded_results\202504-08_AZ_BB_ChainPSKU_OT_run\train_till_28_Feb_2025\ml_results\feature_importance_train_till_28_Feb_2025.csv
downloaded_results\202504-08_AZ_BB_ChainPSKU_OT_run\train_till_30_Apr_2025\ml_results\feature_importance_train_till_30_Apr_2025.csv
downloaded_results\202504-08_AZ_BB_ChainPSKU_OT_run\train_till_30_Jun_2025\ml_results\feature_importance_train_till_30_Jun_2025.csv
downloaded_results\202504-08_AZ_BB_ChainPSKU_OT_run\train_till_31_Mar_2025\ml_results\feature_importance_train_till_31_Mar_2025.csv
downloaded_results\202504-08_AZ_BB_ChainPSKU_OT_run\train_till_31_May_2025\ml_results\feature_importance_train_till_31_May_2025.csv


In [87]:
feature_importance_df.to_excel('202505-AZ_BB_OT_Feature_Imp.xlsx', index=False)

In [11]:
collate_file('prophet_data_train_till_').to_excel('202504-08_Prophet_File.xlsx', index=False)

downloaded_results\ECOM_ChainPSKU_OT_run\train_till_28_Feb_2025\prophet_results\prophet_data_train_till_28_Feb_2025.csv
downloaded_results\ECOM_ChainPSKU_OT_run\train_till_30_Apr_2025\prophet_results\prophet_data_train_till_30_Apr_2025.csv
downloaded_results\ECOM_ChainPSKU_OT_run\train_till_30_Jun_2025\prophet_results\prophet_data_train_till_30_Jun_2025.csv
downloaded_results\ECOM_ChainPSKU_OT_run\train_till_31_Mar_2025\prophet_results\prophet_data_train_till_31_Mar_2025.csv
downloaded_results\ECOM_ChainPSKU_OT_run\train_till_31_May_2025\prophet_results\prophet_data_train_till_31_May_2025.csv


### Secondary

In [87]:
sec_query = """SELECT 
    chain,
    parent_material_code, 
    month_date, 
    SUM(sec_actuals_vol_rum_month) AS sec_vol_actuals_rum_month,
    SUM(sec_apo_plan_vol_rum_month) AS sec_apo_plan_vol_rum_month
FROM (
    SELECT 
        month_date, 
        distributor_code, 
        material_code, 
        sec_actuals_vol_rum_month, 
        sec_apo_plan_vol_rum_month
    FROM 
        dwh_bpm_dist_brand_mth_sbp 
    WHERE 
        month_date BETWEEN '2022-04-01' AND '2025-12-31'
) A
JOIN (
    SELECT DISTINCT
        customer, 
        chain
    FROM 
        mst_chain_master 
    WHERE 
        chain_type = 'E Com B2C'
) CC
    ON A.distributor_Code = CC.customer
JOIN (
    SELECT 
        material_code, 
        parent_material_code 
    FROM 
        mst_material 
    WHERE 
        company_code = 'MIL' 
        AND latest_record_ind = 1
) M 
    ON A.material_code = M.material_code
GROUP BY 
    chain,
    parent_material_code, 
    month_date
ORDER BY 
    chain,
    parent_material_code, 
    month_date;
"""


results = pd.read_sql(con=prod_conn, sql=sec_query)
sales_data = pd.DataFrame(results)
sales_data.columns = sales_data.columns.str.lower()
sales_data = sales_data.rename(columns={'parent_material1_code':'parent_material_code'})

In [88]:
sales_data['month_date'] = pd.to_datetime(sales_data['month_date'])

In [89]:
sales_data['chain'] = sales_data['chain'].replace({
    'Grofers': 'Blinkit',
    'Amazon B2C': 'Amazon ARIPL',
    'Flipkart-Grocery': 'Flipkart Grocery',
    'FlipkartGrocery': 'Flipkart Grocery',
    'Flipkart-National': 'Flipkart National',
    'RK WORLDINFOCOM': 'Amazon RK',
    'ZEPTO': 'Zepto',
    'Kiranakart Technologies': 'Zepto',
    'Big basket B2B': 'Big Basket',
    'Big basket B2C': 'Big Basket',
    'Myntra': 'MYNTRA'
})

In [90]:
chains = ['Big Basket', 'Blinkit', 'Amazon ARIPL', 'Flipkart Grocery',
       'Flipkart National', 'Swiggy', 'Zepto', 'Amazon RK', 'City Mall',
       '1MG', 'Dealshare', 'Meesho', 'Nykaa', 'First Cry', 'MYNTRA',
       'Purplle']

In [91]:
for chain in chains:
    if chain not in sales_data['chain'].unique():
        print(chain)

In [92]:
dev_conn = get_dbconnection('DEV')
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment_2""",
    dev_conn
)

realignment_df.columns = realignment_df.columns.str.lower()

def realign_pskus(data, channel='ECOM'):
    realignment_data = realignment_df.copy()
    realignment_data = realignment_data[
        (realignment_data["channel"] == channel)
        | (realignment_data["channel"] == channel + " B2C")
        | (realignment_data["channel"] == "All")
    ]
    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']
    
    data["parent_material_code"] = data["parent_material_code"].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data['parent_material_code'] == old_psku, "parent_material_code"
        ] = new_psku

    return data


Credentials retrieved successfully for dev db.


In [93]:
realigned_df = realign_pskus(sales_data.copy())

In [94]:
realigned_df = realigned_df.groupby(
    ['chain', 'parent_material_code', 'month_date'], as_index=False
).sum()

In [95]:
old_pskus = realignment_df[
    (realignment_df["channel"] == "ECOM")
    | (realignment_df["channel"] == "ECOM" + " B2C")
    | (realignment_df["channel"] == "All")
]['psku old'].unique()

for psku in old_pskus:
    assert psku not in realigned_df['parent_material_code'].unique()

In [96]:
realigned_df['key'] = realigned_df['chain'] + '_' + realigned_df['parent_material_code'].astype(str) 
realigned_df['month_date'] = pd.to_datetime(realigned_df['month_date'])

realigned_df.duplicated(subset=['key', 'month_date']).sum()

0

In [97]:
realigned_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()

0

In [99]:
realigned_df


,chain,parent_material_code,month_date,sec_vol_actuals_rum_month,sec_apo_plan_vol_rum_month,key
0,1MG,715095,2024-01-31,0.0,0.000,1MG_715095
1,1MG,715096,2023-05-31,0.0,0.000,1MG_715096
2,1MG,715096,2023-06-30,0.0,0.189,1MG_715096
3,1MG,715096,2023-07-31,0.0,0.016,1MG_715096
4,1MG,715096,2023-08-31,0.0,0.000,1MG_715096
...,...,...,...,...,...,...
1081559,imli,807033,2025-04-30,0.0,0.000,imli_807033
1081560,imli,807033,2025-05-31,0.0,0.000,imli_807033
1081561,imli,807033,2025-06-30,0.0,0.000,imli_807033
1081562,imli,807033,2025-07-31,0.0,0.000,imli_807033


In [105]:
trend_file_df['platform_name'].unique()

array(['Amazon', 'Big Basket', 'Flipkart Grocery', 'Flipkart National'],
      dtype=object)

In [106]:
trend_file_df['platform_updated'] = np.where(
    trend_file_df['platform_name'] == 'Amazon', 
    np.where(
        trend_file_df['portfolio'].isin(['Foods', 'Saffola Oils']), 
        'Amazon ARIPL', 
        'Amazon RK'
    ),
    trend_file_df['platform_name']
)

In [108]:
trend_file_df['platform_updated'].unique()

array(['Amazon RK', 'Big Basket', 'Flipkart Grocery', 'Flipkart National',
       'Amazon ARIPL'], dtype=object)

In [109]:
trend_file_df.groupby(
    'platform_name'
)['platform_updated'].unique()

platform_name
Amazon               [Amazon RK, Amazon ARIPL]
Big Basket                        [Big Basket]
Flipkart Grocery            [Flipkart Grocery]
Flipkart National          [Flipkart National]
Name: platform_updated, dtype: object

In [102]:
trend_file_df['portfolio']

0             Hair Oils
1             Hair Oils
2             Hair Oils
3             Hair Oils
4             Hair Oils
              ...      
120441    Male Grooming
120442    Male Grooming
120443    Male Grooming
120444    Male Grooming
120445    Male Grooming
Name: portfolio, Length: 120446, dtype: object

In [113]:
trend_file_df['key'] = trend_file_df[
    ['platform_updated', 'parent_material_code']].astype(str).agg('_'.join, axis=1)

In [116]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    realigned_df[['key', 'month_date', 'sec_vol_actuals_rum_month']],
    on=['key', 'month_date'],
    how='left'
)
assert len_before_merge == len(trend_file_df)
del len_before_merge